# Tests de mini-funcionalidades de import_trips_from_dataframe

Este notebook se usa para probar helpers y bloques internos de `import_trips_from_dataframe()` antes de hacer tests más integrados de la función completa.

Objetivo:
- verificar minifuncionalidades de forma aislada
- detectar errores de implementación temprano
- dejar una base fácil de portar después a pytest

Convenciones:
- los tests usan `assert`
- cuando una prueba necesita inspección visual, se acompaña con `display(...)`
- las pruebas de este notebook no reemplazan tests integrados ni validate_trips

## Sección 0. Preparación
- Imports
- Carga del módulo
- Helpers de apoyo para tests del notebook
- Convenciones del notebook
- (Opcional) función para resetear/revisar environment

In [217]:
import json
import math
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import h3

In [218]:
from pylondrina.schema import (
    DomainSpec,
    FieldSpec,
    TripSchema,
    TripSchemaEffective,
)

from pylondrina.importing import (
    ImportOptions,
    import_trips_from_dataframe,
)

from pylondrina.datasets import TripDataset
from pylondrina.reports import ImportReport, Issue
from pylondrina.errors import ImportError as PylondrinaImportError, SchemaError, PylondrinaError

In [219]:
from pylondrina.importing import (
    _normalize_options,
    _check_schema_for_import,
    _apply_field_correspondence,
    _apply_value_correspondence,
    _get_unknown_token,
    _is_already_correct_dtype,
    _coerce_series_to_dtype,
    _coerce_columns_by_dtype,
    _normalize_source_timezone,
    _normalize_datetime_column,
    _normalize_datetime_columns,
    _normalize_hhmm_series,
    _normalize_tier2_hhmm_columns,
    _dm_to_dd,
    _dms_to_dd,
    _parse_coord_value,
    _parse_od_coordinate_columns,
    _derive_h3_indices,
    _build_keep_schema_fields,
    _select_final_columns,
    _final_required_check,
    _prune_schema_effective,
    _build_issues_summary,
    _build_import_metadata,
    _build_import_report,
    _first_required_check_and_temporal_tier,
    _standardize_categorical_values,
    _ensure_movement_id,
    _ensure_single_stage_ids,   
)

In [220]:
from pylondrina.issues.core import emit_issue, emit_and_maybe_raise
from pylondrina.issues.catalog_import_trips import IMPORT_ISSUES

#### Helpers de apoyo para los tests

In [221]:
# Helper para mostrar exito simple
def show_ok(label: str):
    print(f"OK - {label}")

# Helper para serializacion JSON-safe: Util para metadata, parameters y report
def assert_json_safe(obj, label: str = "object"):
    try:
        json.dumps(obj, default=str)
    except Exception as e:
        raise AssertionError(f"{label} no es JSON-safe: {e}") from e

# Helper para comparar columnas del dataframe: Muy util para seleccion final, IDs, etc.
def assert_columns_equal(df: pd.DataFrame, expected_cols, label: str = "columns"):
    actual = list(df.columns)
    expected = list(expected_cols)
    assert set(actual) == set(expected), f"{label} distintas.\nActual:   {actual}\nExpected: {expected}"

# Helpers (get_issue_codes, assert_issue_present, assert_issue_absent) para verificar issues por code
def get_issue_codes(issues):
    return [i.code if hasattr(i, "code") else i.get("code") for i in issues]

def assert_issue_present(issues, code: str):
    codes = get_issue_codes(issues)
    assert code in codes, f"No se encontró issue {code!r}. Codes presentes: {codes}"

def assert_issue_absent(issues, code: str):
    codes = get_issue_codes(issues)
    assert code not in codes, f"Se encontró issue inesperado {code!r}. Codes presentes: {codes}"

# Helper para revisar dtype de columna
def assert_dtype(df: pd.DataFrame, col: str, expected_dtype: str):
    actual = str(df[col].dtype)
    assert actual == expected_dtype, f"Columna {col!r}: dtype {actual!r} != {expected_dtype!r}"

# Helper para revisar NA counts
def assert_na_count(df: pd.DataFrame, col: str, expected: int):
    actual = int(df[col].isna().sum())
    assert actual == expected, f"Columna {col!r}: n_na {actual} != {expected}"

#### Configuración visual y smoke de imports

In [222]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 120)

print("Imports OK")
show_ok("Sección 0 cargada")

Imports OK
OK - Sección 0 cargada


## Sección 1. Utilidades puras
### Grupo 1 - utilidades base
En esta sección se prueban helpers puras o casi puras que sirven como base para bloques más grandes del import.

Objetivo:
- verificar outputs exactos,
- cubrir casos felices, bordes válidos, bordes inválidos y nulos,
- detectar errores temprano antes de probar helpers de integración.

otros
- source timezone
- unknown token
- detección de dtype correcto
- parseo de coordenadas DD/DM/DMS
- normalización HH:MM
- build_keep_schema_fields
- build_issues_summary

In [223]:
# Test de _normalize_source_timezone

tz_cases = [
    (None, (None, "none")),
    ("", (None, "empty")),
    ("   ", (None, "empty")),
    ("UTC", ("UTC", "utc")),
    ("utc", ("UTC", "utc")),
    ("Z", ("UTC", "utc")),
    ("-03:00", ("-03:00", "offset")),
    ("+00:00", ("+00:00", "offset")),
    ("America/Santiago", ("America/Santiago", "iana")),
    ("  America/Santiago  ", ("America/Santiago", "iana")),
    ("Santiago/Chile", (None, "invalid")),
    ("abc", (None, "invalid")),
]

for raw, expected in tz_cases:
    got = _normalize_source_timezone(raw)
    assert got == expected, f"_normalize_source_timezone({raw!r}) -> {got}, esperado {expected}"

show_ok("_normalize_source_timezone")

OK - _normalize_source_timezone


In [224]:
# Test de _get_unknown_token

domain_none = None
domain_unknown = DomainSpec(values=["bus", "unknown", "metro"], extendable=False, aliases=None)
domain_other = DomainSpec(values=["bus", "other", "metro"], extendable=False, aliases=None)
domain_OTHER = DomainSpec(values=["bus", "OTHER", "metro"], extendable=False, aliases=None)
domain_Unknown = DomainSpec(values=["bus", "Unknown", "metro"], extendable=False, aliases=None)
domain_no_candidate = DomainSpec(values=["bus", "metro"], extendable=False, aliases=None)
domain_empty = DomainSpec(values=[], extendable=False, aliases=None)

assert _get_unknown_token(domain_none) == "unknown"
assert _get_unknown_token(domain_unknown) == "unknown"
assert _get_unknown_token(domain_other) == "other"
assert _get_unknown_token(domain_OTHER) == "OTHER"
assert _get_unknown_token(domain_Unknown) == "Unknown"
assert _get_unknown_token(domain_no_candidate) == "unknown"
assert _get_unknown_token(domain_empty) == "unknown"

show_ok("_get_unknown_token")

OK - _get_unknown_token


In [225]:
# Test de _is_already_correct_dtype

s_string = pd.Series(["a", "b", pd.NA], dtype="string")
s_int = pd.Series([1, 2, pd.NA], dtype="Int64")
s_float = pd.Series([1.5, 2.5, np.nan], dtype="float64")
s_bool = pd.Series([True, False, pd.NA], dtype="boolean")
s_dt_naive = pd.Series(pd.to_datetime(["2026-03-01 08:00:00", None]))
s_dt_utc = pd.Series(pd.to_datetime(["2026-03-01T08:00:00Z", None], utc=True))
s_obj = pd.Series(["a", "b", None], dtype="object")

assert _is_already_correct_dtype(s_string, "string") is True
assert _is_already_correct_dtype(s_string, "categorical") is True
assert _is_already_correct_dtype(s_int, "int") is True
assert _is_already_correct_dtype(s_float, "float") is True
assert _is_already_correct_dtype(s_bool, "bool") is True
assert _is_already_correct_dtype(s_dt_naive, "datetime") is True
assert _is_already_correct_dtype(s_dt_utc, "datetime") is True

assert _is_already_correct_dtype(s_obj, "string") is False
assert _is_already_correct_dtype(s_string, "int") is False
assert _is_already_correct_dtype(s_int, "float") is False
assert _is_already_correct_dtype(s_float, "bool") is False
assert _is_already_correct_dtype(s_bool, "datetime") is False

show_ok("_is_already_correct_dtype")

OK - _is_already_correct_dtype


In [226]:
# Tests de _dm_to_dd y _dms_to_dd

# DM
assert math.isclose(_dm_to_dd(33, 30, "N"), 33.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dm_to_dd(70, 30, "W"), -70.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dm_to_dd(33, 27.6, "S"), -(33 + 27.6/60.0), rel_tol=0, abs_tol=1e-9)

# DMS
assert math.isclose(_dms_to_dd(33, 30, 0, "N"), 33.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dms_to_dd(70, 30, 0, "W"), -70.5, rel_tol=0, abs_tol=1e-9)
assert math.isclose(_dms_to_dd(33, 27, 36, "S"), -(33 + 27/60.0 + 36/3600.0), rel_tol=0, abs_tol=1e-9)

show_ok("_dm_to_dd / _dms_to_dd")

OK - _dm_to_dd / _dms_to_dd


In [227]:
# Tests de _parse_coord_value

val, status = _parse_coord_value(np.nan)
assert np.isnan(val)
assert status == "null"

val, status = _parse_coord_value(None)
assert np.isnan(val)
assert status == "null"

val, status = _parse_coord_value(10)
assert val == 10.0
assert status == "numeric"

val, status = _parse_coord_value(-33.45)
assert math.isclose(val, -33.45, rel_tol=0, abs_tol=1e-12)
assert status == "numeric"

val, status = _parse_coord_value("-33.446160")
assert math.isclose(val, -33.446160, rel_tol=0, abs_tol=1e-12)
assert status == "dd_direct"

val, status = _parse_coord_value("335208,7188")
assert math.isclose(val, 335208.7188, rel_tol=0, abs_tol=1e-9)
assert status == "dd_comma_decimal"

val, status = _parse_coord_value("33 27.0000 S")
assert math.isclose(val, -33.45, rel_tol=0, abs_tol=1e-9)
assert status == "dm"

dms_tests = {
    '33° 27\' 00" S': -33.45,
    "33 27 00 S" : -33.45,
    "70 30 00 W" : -70.5,
    '12 05 30.5 N' : 12.091805556,
}

for t, e in dms_tests.items():
    val, status = _parse_coord_value(t)
    assert math.isclose(val, e, rel_tol=0, abs_tol=1e-9)
    assert status == "dms"

val, status = _parse_coord_value("")
assert np.isnan(val)
assert status == "empty"

val, status = _parse_coord_value("   ")
assert np.isnan(val)
assert status == "empty"

val, status = _parse_coord_value("abc")
assert np.isnan(val)
assert status == "unparsed"

show_ok("_parse_coord_value")

OK - _parse_coord_value


In [228]:
# Tests de _normalize_hhmm_series

s_hhmm = pd.Series(
    ["08:23", "12:30", "24:00", "ab:cd", "", None, "7:05", "09:60", " 07:05 ", "00:00", "23:59"],
    dtype="object",
)

out_hhmm, stats_hhmm = _normalize_hhmm_series(s_hhmm)

assert_dtype(pd.DataFrame({"hhmm": out_hhmm}), "hhmm", "string")

# válidos preservados
assert out_hhmm.iloc[0] == "08:23"
assert out_hhmm.iloc[1] == "12:30"
assert out_hhmm.iloc[8] == "07:05"
assert out_hhmm.iloc[9] == "00:00"
assert out_hhmm.iloc[10] == "23:59"

# inválidos a NA
assert pd.isna(out_hhmm.iloc[2])   # 24:00
assert pd.isna(out_hhmm.iloc[3])   # ab:cd
assert pd.isna(out_hhmm.iloc[4])   # ""
assert pd.isna(out_hhmm.iloc[5])   # None
assert pd.isna(out_hhmm.iloc[6])   # 7:05
assert pd.isna(out_hhmm.iloc[7])   # 09:60

assert stats_hhmm["n_total"] == 11
assert stats_hhmm["n_invalid"] == 4   # 24:00, ab:cd, 7:05, 09:60
assert stats_hhmm["n_na"] == 6        # inválidos + vacíos/nulos

show_ok("_normalize_hhmm_series")

OK - _normalize_hhmm_series


In [229]:
# Tests de _build_keep_schema_fields

schema_g1 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
    },
    required=["movement_id", "trip_id"],
    semantic_rules=None,
)

# selected_fields=None -> todos los fields del schema
res_none = _build_keep_schema_fields(schema_g1, None)
assert res_none["schema_fields"] == {"movement_id", "trip_id", "movement_seq", "purpose", "mode"}
assert res_none["required_fields"] == {"movement_id", "trip_id"}
assert res_none["keep_schema_fields"] == {"movement_id", "trip_id", "movement_seq", "purpose", "mode"}

# selected_fields=[] -> solo required
res_empty = _build_keep_schema_fields(schema_g1, [])
assert res_empty["keep_schema_fields"] == {"movement_id", "trip_id"}

# subconjunto válido -> required ∪ selected
res_subset = _build_keep_schema_fields(schema_g1, ["purpose", "mode"])
assert res_subset["keep_schema_fields"] == {"movement_id", "trip_id", "purpose", "mode"}

# selected_fields inválido -> ValueError
try:
    _build_keep_schema_fields(schema_g1, ["purpose", "fake_field"])
    assert False, "Debió fallar por selected_fields inválido"
except ValueError:
    pass

show_ok("_build_keep_schema_fields")

OK - _build_keep_schema_fields


In [230]:
# Tests de _build_issues_summary

# caso vacío
summary_empty = _build_issues_summary([])
assert summary_empty == {"counts": {}, "by_code": {}}

issues_g1 = [
    Issue(level="warning", code="CODE_A", message="a"),
    Issue(level="warning", code="CODE_A", message="a2"),
    Issue(level="info", code="CODE_B", message="b"),
    Issue(level="error", code="CODE_C", message="c"),
]

summary_g1 = _build_issues_summary(issues_g1)

assert summary_g1["counts"] == {
    "warning": 2,
    "info": 1,
    "error": 1,
}

assert summary_g1["by_code"] == {
    "CODE_A": 2,
    "CODE_B": 1,
    "CODE_C": 1,
}

assert_json_safe(summary_g1, "issues_summary")

show_ok("_build_issues_summary")

OK - _build_issues_summary


In [231]:
show_ok("Grupo 1 completo")

OK - Grupo 1 completo


## Sección 2. Schema y options
### Grupo 2A - detección temporal

En esta subsección se prueba la detección del tier temporal y el primer chequeo estructural mínimo asociado a tiempos.

Objetivo:
- verificar que `_first_required_check_and_temporal_tier(...)` detecte correctamente `tier_1`, `tier_2` y `tier_3`,
- comprobar que emite los issues esperados,
- y confirmar que distingue entre campos faltantes derivables y no derivables.

In [232]:
# Tests de _first_required_check_and_temporal_tier

schema_temporal = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
        "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string", required=False),
        "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string", required=False),
    },
    required=["user_id", "movement_id"],
    semantic_rules=None,
)

# Caso 1: Tier 1 con UTC explícito
df_t1 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_time_utc": ["2026-03-01T08:00:00Z", "2026-03-01T09:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z", "2026-03-01T09:40:00Z"],
})

issues_t1, tier_t1, fields_t1 = _first_required_check_and_temporal_tier(
    df_t1,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t1 == "tier_1", f"Esperaba tier_1 y obtuvo {tier_t1!r}"
assert set(fields_t1) == {"origin_time_utc", "destination_time_utc"}
assert_issue_absent(issues_t1, "IMP.TEMPORAL.TIER_LIMITED")
# movement_id falta pero es derivable, por lo tanto no debe abortar ni emitir missing_required fatal
assert_issue_absent(issues_t1, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 2: Tier 2 con HH:MM
df_t2 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_time_local_hhmm": ["08:00", "09:00"],
    "destination_time_local_hhmm": ["08:30", "09:40"],
})

issues_t2, tier_t2, fields_t2 = _first_required_check_and_temporal_tier(
    df_t2,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t2 == "tier_2", f"Esperaba tier_2 y obtuvo {tier_t2!r}"
assert set(fields_t2) == {"origin_time_local_hhmm", "destination_time_local_hhmm"}
assert_issue_present(issues_t2, "IMP.TEMPORAL.TIER_LIMITED")
assert_issue_absent(issues_t2, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 3: Tier 3 sin información temporal OD
df_t3 = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "trip_id": ["t1", "t2"],
})

issues_t3, tier_t3, fields_t3 = _first_required_check_and_temporal_tier(
    df_t3,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_t3 == "tier_3", f"Esperaba tier_3 y obtuvo {tier_t3!r}"
assert fields_t3 == []
assert_issue_present(issues_t3, "IMP.TEMPORAL.TIER_LIMITED")
assert_issue_absent(issues_t3, "IMP.INPUT.MISSING_REQUIRED_FIELD")

# Caso 4: DataFrame vacío -> issue EMPTY_DATAFRAME
df_empty = pd.DataFrame(columns=["user_id", "origin_time_utc", "destination_time_utc"])

issues_empty, tier_empty, fields_empty = _first_required_check_and_temporal_tier(
    df_empty,
    schema=schema_temporal,
    single_stage=False,
    strict=False,
)

assert tier_empty == "tier_1", f"Esperaba tier_1 por columnas presentes y obtuvo {tier_empty!r}"
assert set(fields_empty) == {"origin_time_utc", "destination_time_utc"}
assert_issue_present(issues_empty, "IMP.INPUT.EMPTY_DATAFRAME")

# Caso 5: Falta required no derivable -> issue MISSING_REQUIRED_FIELD
df_missing_non_derivable = pd.DataFrame({
    "origin_time_utc": ["2026-03-01T08:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z"],
})

try:
    issues_missing, tier_missing, fields_missing = _first_required_check_and_temporal_tier(
        df_missing_non_derivable,
        schema=schema_temporal,
        single_stage=False,
        strict=False,
    )
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_first_required_check_and_temporal_tier")

OK - _first_required_check_and_temporal_tier


### Grupo 3 - schema usable para import

En esta sección se prueba la helper `_check_schema_for_import(...)`.

Objetivo:
- verificar cuándo el schema permite seguir con import,
- distinguir entre degradaciones con warning y abortos fatales,
- y revisar cómo se construyen `dtype_effective` y `overrides` en `TripSchemaEffective`.

In [233]:
# Tests de _check_schema_for_import -> caso feliz mínimo

schema_g3_ok = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", "study"], extendable=True, aliases=None),
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_ok, schema_effective_ok, schema_version_ok = _check_schema_for_import(schema_g3_ok)

assert issues_ok == []
assert schema_version_ok == "0.1.0"
assert schema_effective_ok.dtype_effective == {
    "user_id": "string",
    "purpose": "categorical",
}
assert schema_effective_ok.overrides == {}

show_ok("_check_schema_for_import - caso feliz mínimo")

OK - _check_schema_for_import - caso feliz mínimo


In [234]:
# Tests de _check_schema_for_import -> fields vacío

schema_g3_empty_fields = TripSchema(
    version="0.1.0",
    fields={},
    required=["user_id"],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_empty_fields)
    assert False, "Debió lanzar PylondrinaSchemaError por fields vacío"
except SchemaError as error:
    assert_issue_present(error.issues, "SCH.TRIP_SCHEMA.EMPTY_FIELDS")

show_ok("_check_schema_for_import - fields vacío")

OK - _check_schema_for_import - fields vacío


In [235]:
# Tests de _check_schema_for_import -> required vacío

schema_g3_empty_required = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
    },
    required=[],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_empty_required)
    assert False, "Debió lanzar PylondrinaSchemaError por required vacío"
except SchemaError as error:
    assert_issue_present(error.issues, "SCH.TRIP_SCHEMA.EMPTY_REQUIRED")

show_ok("_check_schema_for_import - required vacío")

OK - _check_schema_for_import - required vacío


In [236]:
# Tests de _check_schema_for_import -> required fuera de fields

schema_g3_required_outside = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
    },
    required=["user_id", "trip_id"],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_required_outside)
    assert False, "Debió lanzar PylondrinaSchemaError por required fuera de fields"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_check_schema_for_import - required fuera de fields")

OK - _check_schema_for_import - required fuera de fields


In [237]:
# Tests de _check_schema_for_import -> dtype desconocido

schema_g3_unknown_dtype = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "weird_field": FieldSpec(name="weird_field", dtype="vector", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_unknown_dtype, schema_effective_unknown_dtype, _ = _check_schema_for_import(schema_g3_unknown_dtype)

assert_issue_present(issues_unknown_dtype, "SCH.FIELD_SPEC.UNKNOWN_DTYPE")
assert schema_effective_unknown_dtype.dtype_effective["user_id"] == "string"
assert schema_effective_unknown_dtype.dtype_effective["weird_field"] == "string"
assert "weird_field" in schema_effective_unknown_dtype.overrides
assert "dtype_invalid" in schema_effective_unknown_dtype.overrides["weird_field"]["reasons"]

show_ok("_check_schema_for_import - dtype desconocido")

OK - _check_schema_for_import - dtype desconocido


In [238]:
# Tests de _check_schema_for_import -> categórico sin domain

schema_g3_cat_no_domain = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False, domain=None),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_cat_no_domain, schema_effective_cat_no_domain, _ = _check_schema_for_import(schema_g3_cat_no_domain)

assert_issue_present(issues_cat_no_domain, "SCH.DOMAIN.MISSING_FOR_CATEGORICAL")
assert schema_effective_cat_no_domain.dtype_effective["purpose"] == "string"
assert "purpose" in schema_effective_cat_no_domain.overrides
assert "categorical_no_domain" in schema_effective_cat_no_domain.overrides["purpose"]["reasons"]

show_ok("_check_schema_for_import - categórico sin domain")

OK - _check_schema_for_import - categórico sin domain


In [239]:
# Tests de _check_schema_for_import -> domain vacío

schema_g3_empty_domain = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True, aliases=None),
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_empty_domain, schema_effective_empty_domain, _ = _check_schema_for_import(schema_g3_empty_domain)

assert_issue_present(issues_empty_domain, "SCH.DOMAIN.EMPTY_VALUES")
assert schema_effective_empty_domain.dtype_effective["purpose"] == "categorical"
assert schema_effective_empty_domain.overrides == {}

show_ok("_check_schema_for_import - domain vacío")

OK - _check_schema_for_import - domain vacío


In [240]:
# Tests de _check_schema_for_import -> values no string

schema_g3_non_string_values = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", 123, None], extendable=True, aliases=None),
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_non_string_values, schema_effective_non_string_values, _ = _check_schema_for_import(schema_g3_non_string_values)

assert_issue_present(issues_non_string_values, "SCH.DOMAIN.NON_STRING_VALUES")
assert schema_effective_non_string_values.dtype_effective["purpose"] == "categorical"

show_ok("_check_schema_for_import - values no string")

OK - _check_schema_for_import - values no string


In [241]:
# Tests de _check_schema_for_import -> constraints desconocidas

schema_g3_unknown_constraint = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(
            name="user_id",
            dtype="string",
            required=True,
            constraints={"min_foo": 10},
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_unknown_constraint)
    assert False, "Debió lanzar PylondrinaSchemaError por constraint desconocida"
except SchemaError as error:
    assert_issue_present(error.issues, "SCH.CONSTRAINTS.UNKNOWN_RULE")

show_ok("_check_schema_for_import - constraints desconocidas")

OK - _check_schema_for_import - constraints desconocidas


In [242]:
# Tests de _check_schema_for_import -> pattern inválido

schema_g3_bad_pattern = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(
            name="user_id",
            dtype="string",
            required=True,
            constraints={"pattern": "["},  # regex inválida
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_bad_pattern)
    assert False, "Debió lanzar PylondrinaSchemaError por pattern inválido"
except SchemaError as error:
    assert_issue_present(error.issues, "SCH.CONSTRAINTS.INVALID_FORMAT")

show_ok("_check_schema_for_import - pattern inválido")

OK - _check_schema_for_import - pattern inválido


In [243]:
# Tests de _check_schema_for_import -> constraints con formato inválido

schema_g3_bad_constraints_format = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(
            name="user_id",
            dtype="string",
            required=True,
            constraints=["not", "a", "dict"],  # formato inválido
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

try:
    _check_schema_for_import(schema_g3_bad_constraints_format)
    assert False, "Debió lanzar PylondrinaSchemaError por constraints con formato inválido"
except SchemaError as error:
    assert_issue_present(error.issues, "SCH.CONSTRAINTS.INVALID_FORMAT")

show_ok("_check_schema_for_import - constraints formato inválido")

OK - _check_schema_for_import - constraints formato inválido


In [244]:
# Tests de _check_schema_for_import -> constraint incompatible con dtype

schema_g3_incompatible_constraint = TripSchema(
    version="0.1.0",
    fields={
        "trip_count": FieldSpec(
            name="trip_count",
            dtype="int",
            required=True,
            constraints={"pattern": r"^\d+$"},  # pattern en int -> incompatible
        ),
    },
    required=["trip_count"],
    semantic_rules=None,
)

issues_incompatible_constraint, schema_effective_incompatible_constraint, _ = _check_schema_for_import(
    schema_g3_incompatible_constraint
)

assert_issue_present(issues_incompatible_constraint, "SCH.CONSTRAINTS.INCOMPATIBLE_WITH_DTYPE")
assert schema_effective_incompatible_constraint.dtype_effective["trip_count"] == "int"

show_ok("_check_schema_for_import - constraint incompatible con dtype")

OK - _check_schema_for_import - constraint incompatible con dtype


In [245]:
# Test integrado pequeño del Grupo 3

schema_g3_integrated = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "weird_field": FieldSpec(name="weird_field", dtype="vector", required=False),
        "cat_no_domain": FieldSpec(name="cat_no_domain", dtype="categorical", required=False, domain=None),
        "cat_empty_domain": FieldSpec(
            name="cat_empty_domain",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=[], extendable=True, aliases=None),
        ),
        "cat_non_string": FieldSpec(
            name="cat_non_string",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["ok", 1, None], extendable=True, aliases=None),
        ),
    },
    required=["user_id"],
    semantic_rules=None,
)

issues_integrated_g3, schema_effective_integrated_g3, version_integrated_g3 = _check_schema_for_import(
    schema_g3_integrated
)

assert version_integrated_g3 == "0.1.0"

assert_issue_present(issues_integrated_g3, "SCH.FIELD_SPEC.UNKNOWN_DTYPE")
assert_issue_present(issues_integrated_g3, "SCH.DOMAIN.MISSING_FOR_CATEGORICAL")
assert_issue_present(issues_integrated_g3, "SCH.DOMAIN.EMPTY_VALUES")
assert_issue_present(issues_integrated_g3, "SCH.DOMAIN.NON_STRING_VALUES")

assert schema_effective_integrated_g3.dtype_effective["weird_field"] == "string"
assert schema_effective_integrated_g3.dtype_effective["cat_no_domain"] == "string"
assert schema_effective_integrated_g3.dtype_effective["cat_empty_domain"] == "categorical"
assert schema_effective_integrated_g3.dtype_effective["cat_non_string"] == "categorical"

assert "dtype_invalid" in schema_effective_integrated_g3.overrides["weird_field"]["reasons"]
assert "categorical_no_domain" in schema_effective_integrated_g3.overrides["cat_no_domain"]["reasons"]

assert_json_safe(schema_effective_integrated_g3.to_dict(), "schema_effective_integrated_g3")

show_ok("Grupo 3 - test integrado pequeño")

OK - Grupo 3 - test integrado pequeño


In [246]:
show_ok("Grupo 3 completo")

OK - Grupo 3 completo


## Sección 3. Mappings y categóricos
### Grupo 4 - field correspondence

En esta sección se prueba la helper `_apply_field_correspondence(...)`.

Objetivo:
- verificar el renombrado estructural de columnas fuente a campos canónicos,
- comprobar los casos de identidad canónica,
- revisar errores por mapping inválido o conflictivo,
- y asegurar que los campos opcionales faltantes emitan los issues esperados.

In [247]:
# Fixtures comunes del Grupo 4

schema_g4 = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "dummy_id": FieldSpec(name="dummy_id", dtype="string", required=False),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=False),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_g4_base = pd.DataFrame({
    "uid": ["u1", "u2"],
    "motivo": ["trabajo", "estudio"],
    "modo": ["bus", "metro"],
    "o_lat": [-33.45, -33.46],
    "o_lon": [-70.60, -70.61],
    "raw_extra": ["A", "B"],
})

In [248]:
# Tests de _apply_field_correspondence -> mapping válido

field_corr_valid = {
    "user_id": "uid",
    "purpose": "motivo",
    "mode": "modo",
    "origin_latitude": "o_lat",
    "origin_longitude": "o_lon",
}

work_valid, applied_valid, issues_valid = _apply_field_correspondence(
    df_g4_base.copy(deep=True),
    schema=schema_g4,
    field_correspondence=field_corr_valid,
    strict=False,
)

assert_columns_equal(
    work_valid,
    ["user_id", "purpose", "mode", "origin_latitude", "origin_longitude", "raw_extra"],
    "field correspondence válido",
)

assert applied_valid == {
    "user_id": "uid",
    "purpose": "motivo",
    "mode": "modo",
    "origin_latitude": "o_lat",
    "origin_longitude": "o_lon",
}

assert work_valid["user_id"].tolist() == ["u1", "u2"]
assert work_valid["purpose"].tolist() == ["trabajo", "estudio"]
assert work_valid["mode"].tolist() == ["bus", "metro"]
assert work_valid["origin_latitude"].tolist() == [-33.45, -33.46]
assert work_valid["origin_longitude"].tolist() == [-70.60, -70.61]

# Como trip_id es opcional y no está presente ni mapeado ni es derivable obligatorio aquí, se reporta como opcional ausente
display(issues_valid)
assert_issue_present(issues_valid, "IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND")

show_ok("_apply_field_correspondence - mapping válido")

[Issue(level='info', code='IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND', message="El campo opcional 'dummy_id' no está presente en la fuente y no se encontró correspondencia; se omitirá.", field='dummy_id', source_field=None, row_count=None, details={'field': 'dummy_id', 'source_columns': ['uid', 'motivo', 'modo', 'o_lat', 'o_lon', 'raw_extra'], 'field_correspondence_used': False, 'action': 'omit_optional'})]

OK - _apply_field_correspondence - mapping válido


In [249]:
# Tests de _apply_field_correspondence -> identidad canónica

df_g4_identity = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "purpose": ["work", "study"],
    "raw_extra": ["A", "B"],
})

field_corr_identity = {
    "user_id": "user_id",
    "purpose": "purpose",
}

work_identity, applied_identity, issues_identity = _apply_field_correspondence(
    df_g4_identity.copy(deep=True),
    schema=schema_g4,
    field_correspondence=field_corr_identity,
    strict=False,
)

# No debe renombrar nada realmente
assert_columns_equal(
    work_identity,
    ["user_id", "purpose", "raw_extra"],
    "field correspondence identidad",
)

# Las identidades no se incluyen en applied
assert applied_identity == {}

# mode, origin_latitude, origin_longitude son opcionales y faltan
assert_issue_present(issues_identity, "IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND")

show_ok("_apply_field_correspondence - identidad canónica")

OK - _apply_field_correspondence - identidad canónica


In [250]:
# Tests de _apply_field_correspondence -> source faltante requerido

field_corr_missing_required = {
    "user_id": "uid_missing",
}

try:
    _apply_field_correspondence(
        df_g4_base.copy(deep=True),
        schema=schema_g4,
        field_correspondence=field_corr_missing_required,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por source faltante requerido"
except PylondrinaImportError as error:
    print("Mensaje de error: ", error.message)
    assert_issue_present(error.issues, "MAP.FIELDS.MISSING_SOURCE_COLUMN")

show_ok("_apply_field_correspondence - source faltante requerido")

Mensaje de error:  field_correspondence referencia una columna fuente inexistente para 'user_id': 'uid_missing'.
OK - _apply_field_correspondence - source faltante requerido


In [251]:
# Tests de _apply_field_correspondence -> source faltante opcional

field_corr_missing_optional = {
    "user_id": "uid",
    "purpose": "motivo_missing",  # opcional
}

work_missing_optional, applied_missing_optional, issues_missing_optional = _apply_field_correspondence(
    df_g4_base.copy(deep=True),
    schema=schema_g4,
    field_correspondence=field_corr_missing_optional,
    strict=False,
)

assert "user_id" in work_missing_optional.columns
assert "purpose" not in work_missing_optional.columns
assert applied_missing_optional == {"user_id": "uid"}

assert_issue_present(issues_missing_optional, "IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND")

show_ok("_apply_field_correspondence - source faltante opcional")

OK - _apply_field_correspondence - source faltante opcional


In [252]:
# Tests de  _apply_field_correspondence -> canonical inexistente en schema

field_corr_unknown_canonical = {
    "fake_field": "uid",
}

try:
    _apply_field_correspondence(
        df_g4_base.copy(deep=True),
        schema=schema_g4,
        field_correspondence=field_corr_unknown_canonical,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por canonical inexistente"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "MAP.FIELDS.UNKNOWN_CANONICAL_FIELD")

show_ok("_apply_field_correspondence - canonical inexistente en schema")

OK - _apply_field_correspondence - canonical inexistente en schema


In [253]:
# Tests de _apply_field_correspondence -> dos canónicos al mismo source

field_corr_duplicate_target = {
    "purpose": "motivo",
    "mode": "motivo",
}

try:
    _apply_field_correspondence(
        df_g4_base.copy(deep=True),
        schema=schema_g4,
        field_correspondence=field_corr_duplicate_target,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por colisión duplicate target"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "MAP.FIELDS.COLLISION_DUPLICATE_TARGET")

show_ok("_apply_field_correspondence - dos canónicos al mismo source")

OK - _apply_field_correspondence - dos canónicos al mismo source


In [254]:
# Tests de _apply_field_correspondence -> canónico ya presente en conflicto

df_g4_conflict = pd.DataFrame({
    "user_id": ["u_can_1", "u_can_2"],  # ya existe el canónico
    "uid": ["u_src_1", "u_src_2"],      # además intentamos renombrar esto a user_id
    "raw_extra": ["A", "B"],
})

field_corr_conflict = {
    "user_id": "uid",
}

try:
    _apply_field_correspondence(
        df_g4_conflict.copy(deep=True),
        schema=schema_g4,
        field_correspondence=field_corr_conflict,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por conflicto con canónico ya presente"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "MAP.FIELDS.CANONICAL_ALREADY_PRESENT_CONFLICT")

show_ok("_apply_field_correspondence - canónico ya presente en conflicto")

OK - _apply_field_correspondence - canónico ya presente en conflicto


In [255]:
# Tests de _apply_field_correspondence -> opcionales no presentes sin mapping

df_g4_minimal = pd.DataFrame({
    "uid": ["u1", "u2"],
})

field_corr_minimal = {
    "user_id": "uid",
}

work_minimal, applied_minimal, issues_minimal = _apply_field_correspondence(
    df_g4_minimal.copy(deep=True),
    schema=schema_g4,
    field_correspondence=field_corr_minimal,
    strict=False,
)

assert_columns_equal(work_minimal, ["user_id"], "opcionales no presentes sin mapping")
assert applied_minimal == {"user_id": "uid"}

# Como faltan varios opcionales no derivables, debe aparecer OPTIONAL_FIELD_NOT_FOUND
assert_issue_present(issues_minimal, "IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND")

show_ok("_apply_field_correspondence - opcionales no presentes sin mapping")

OK - _apply_field_correspondence - opcionales no presentes sin mapping


In [256]:
# Test integrado pequeño del Grupo 4

df_g4_integrated = pd.DataFrame({
    "user_id": ["u1", "u2"],   # identidad
    "motivo": ["trabajo", "estudio"],
    "modo": ["bus", "metro"],
    "raw_extra": ["A", "B"],
})

field_corr_integrated = {
    "user_id": "user_id",   # identidad
    "purpose": "motivo",    # renombrado real
    "mode": "modo",         # renombrado real
    "origin_latitude": "o_lat_missing",  # opcional faltante
}

work_integrated, applied_integrated, issues_integrated = _apply_field_correspondence(
    df_g4_integrated.copy(deep=True),
    schema=schema_g4,
    field_correspondence=field_corr_integrated,
    strict=False,
)

assert_columns_equal(
    work_integrated,
    ["user_id", "purpose", "mode", "raw_extra"],
    "integrated field correspondence",
)

# identidad no entra en applied
assert applied_integrated == {
    "purpose": "motivo",
    "mode": "modo",
}

assert work_integrated["user_id"].tolist() == ["u1", "u2"]
assert work_integrated["purpose"].tolist() == ["trabajo", "estudio"]
assert work_integrated["mode"].tolist() == ["bus", "metro"]

assert_issue_present(issues_integrated, "IMP.INPUT.OPTIONAL_FIELD_NOT_FOUND")

show_ok("Grupo 4 - test integrado pequeño")

OK - Grupo 4 - test integrado pequeño


In [257]:
show_ok("Grupo 4 completo")

OK - Grupo 4 completo


### Grupo 5 - categóricos y dominios

En esta sección se prueban las helpers de correspondencia de valores y estandarización categórica.

Objetivo:
- verificar que las correspondencias se apliquen solo cuando corresponde,
- comprobar el comportamiento frente a dominios extendibles y no extendibles,
- revisar la interacción con `strict_domains`,
- y asegurar que `domains_effective`, `value_correspondence_applied` y `schema_effective` queden consistentes.

In [258]:
# Tests de _apply_value_correspondence

s_base = pd.Series(["auto", "bus", "auto", pd.NA, "taxi"], dtype="string")

# Caso 1: mapping usado y no usado
s_out_1, used_pairs_1 = _apply_value_correspondence(
    s_base,
    {"auto": "car", "bike": "bicycle"},
)

assert s_out_1.tolist() == ["car", "bus", "car", pd.NA, "taxi"]
assert used_pairs_1 == {"auto": "car"}

# Caso 2: mapping no usado
s_out_2, used_pairs_2 = _apply_value_correspondence(
    s_base,
    {"bike": "bicycle", "train": "rail"},
)

assert s_out_2.equals(s_base)
assert used_pairs_2 == {}

# Caso 3: mapping None
s_out_3, used_pairs_3 = _apply_value_correspondence(s_base, None)

assert s_out_3.equals(s_base)
assert used_pairs_3 == {}

show_ok("_apply_value_correspondence")

OK - _apply_value_correspondence


In [259]:
# Fixtures comunes para _standardize_categorical_values

schema_g5 = TripSchema(
    version="0.1.0",
    fields={
        "mode": FieldSpec(
            name="mode",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=["car", "bus", "unknown"],
                extendable=False,
                aliases=None,
            ),
        ),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=["work", "study"],
                extendable=True,
                aliases=None,
            ),
        ),
        "bootstrap_cat": FieldSpec(
            name="bootstrap_cat",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=[],
                extendable=True,
                aliases=None,
            ),
        ),
        "source_label": FieldSpec(
            name="source_label",
            dtype="string",
            required=False,
            domain=None,
        ),
    },
    required=[],
    semantic_rules=None,
)

schema_effective_g5 = TripSchemaEffective(
    dtype_effective={
        "mode": "categorical",
        "purpose": "categorical",
        "bootstrap_cat": "categorical",
        "source_label": "string",
    },
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

df_g5 = pd.DataFrame({
    "mode": ["auto", "bus", "taxi", pd.NA],
    "purpose": ["trabajo", "study", "funeral", pd.NA],
    "bootstrap_cat": ["alpha", "beta", pd.NA, "alpha"],
    "source_label": ["X", "Y", "Z", "W"],
})

target_schema_fields_all = {"mode", "purpose", "bootstrap_cat", "source_label"}

In [260]:
# Caso principal de _standardize_categorical_values
# Actualizado para la política vigente de dominios vacíos:
# DomainSpec(values=[]) ya no implica extensión automática del dominio.
# Con pocos datos, la inferencia categórica se degrada a string.

value_corr_main = {
    "mode": {
        "auto": "car",      # usado
    #    "bike": "bicycle",  # no usado
    },
    "purpose": {
        "trabajo": "work",  # usado
    },
    "source_label": {
        "X": "foo",         # campo no categórico
    },
}

schema_effective_main = TripSchemaEffective(
    dtype_effective=dict(schema_effective_g5.dtype_effective),
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

work_main, domains_eff_main, vc_applied_main, domains_extended_main, n_map_main, issues_main = (
    _standardize_categorical_values(
        df_g5.copy(deep=True),
        schema=schema_g5,
        schema_effective=schema_effective_main,
        value_correspondence=value_corr_main,
        options=ImportOptions(strict=False, strict_domains=False),
        target_schema_fields=target_schema_fields_all,
    )
)

# mode: auto -> car, taxi -> unknown porque extendable=False
assert work_main["mode"].tolist() == ["car", "bus", "unknown", pd.NA]

# purpose: trabajo -> work, funeral se conserva porque extendable=True y strict_domains=False
assert work_main["purpose"].tolist() == ["work", "study", "funeral", pd.NA]

# bootstrap_cat:
# En la política vigente, domain vacío + pocos datos NO se bootstrappea automáticamente.
# Se conserva como string, pero no se crea domains_effective para el campo.
assert work_main["bootstrap_cat"].tolist() == ["alpha", "beta", pd.NA, "alpha"]
assert "bootstrap_cat" not in domains_eff_main
assert "bootstrap_cat" not in domains_extended_main
assert schema_effective_main.dtype_effective["bootstrap_cat"] == "string"
assert "bootstrap_cat" in schema_effective_main.overrides
assert "categorical_inference_degraded_to_string_high_cardinality" in schema_effective_main.overrides["bootstrap_cat"]["reasons"]

# source_label no debe cambiar porque no es categórico
assert work_main["source_label"].tolist() == ["X", "Y", "Z", "W"]

# mappings realmente usados
assert vc_applied_main == {
    "mode": {"auto": "car"},
    "purpose": {"trabajo": "work"},
}

assert n_map_main == 2

# domains_effective: mode
assert "mode" in domains_eff_main
assert domains_eff_main["mode"]["extended"] is False
assert domains_eff_main["mode"]["unknown_value"] == "unknown"
assert domains_eff_main["mode"]["unknown_values"] == ["taxi"]
assert domains_eff_main["mode"]["added_values"] == []

# domains_effective: purpose
assert "purpose" in domains_eff_main
assert domains_eff_main["purpose"]["extended"] is True
assert domains_eff_main["purpose"]["added_values"] == ["funeral"]
assert "funeral" in domains_eff_main["purpose"]["values"]

# domains_extended
# bootstrap_cat ya no entra aquí, porque no fue extensión de dominio.
assert set(domains_extended_main) == {"purpose"}

# issues esperados
assert_issue_present(issues_main, "DOM.POLICY.FIELD_NOT_EXTENDABLE")
assert_issue_present(issues_main, "DOM.EXTENSION.APPLIED")
assert_issue_present(issues_main, "MAP.VALUES.NON_CATEGORICAL_FIELD")
assert_issue_present(issues_main, "DOM.INFERENCE.DEGRADED_TO_STRING")

assert_json_safe(domains_eff_main, "domains_effective_main")
assert_json_safe(schema_effective_main.to_dict(), "schema_effective_main")

show_ok("_standardize_categorical_values - caso principal actualizado")

OK - _standardize_categorical_values - caso principal actualizado


In [261]:
# Caso específico de _standardize_categorical_values:
# DomainSpec(values=[]) + cardinalidad observada suficientemente baja
# -> inferencia bootstrap controlada del dominio categórico.

df_g5_bootstrap_ok = pd.concat(
    [df_g5.copy(deep=True)] * 10,
    ignore_index=True,
)

# Dejamos bootstrap_cat con 40 valores no nulos y solo 2 categorías.
# Regla vigente:
# n_unique_observed <= alpha * n_rows_non_null
# 2 <= 0.05 * 40
# 2 <= 2.0  -> aplica inferencia
df_g5_bootstrap_ok["bootstrap_cat"] = ["alpha", "beta"] * 20

schema_effective_bootstrap_ok = TripSchemaEffective(
    dtype_effective=dict(schema_effective_g5.dtype_effective),
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

work_bootstrap_ok, domains_eff_bootstrap_ok, vc_applied_bootstrap_ok, domains_extended_bootstrap_ok, n_map_bootstrap_ok, issues_bootstrap_ok = (
    _standardize_categorical_values(
        df_g5_bootstrap_ok.copy(deep=True),
        schema=schema_g5,
        schema_effective=schema_effective_bootstrap_ok,
        value_correspondence={},
        options=ImportOptions(strict=False, strict_domains=False),
        target_schema_fields={"bootstrap_cat"},
    )
)

# La columna se conserva como string con los valores observados.
assert work_bootstrap_ok["bootstrap_cat"].tolist() == ["alpha", "beta"] * 20

# Ahora sí debe existir dominio efectivo para bootstrap_cat.
assert "bootstrap_cat" in domains_eff_bootstrap_ok

bootstrap_domain = domains_eff_bootstrap_ok["bootstrap_cat"]

assert bootstrap_domain["inference_applied"] is True
assert bootstrap_domain["extended"] is False
assert bootstrap_domain["added_values"] == []
assert bootstrap_domain["extended_values"] == []
assert bootstrap_domain["unknown_values"] == []
assert bootstrap_domain["n_unique_observed"] == 2

assert set(bootstrap_domain["observed_values"]) == {"alpha", "beta"}
assert set(bootstrap_domain["values"]) == {"alpha", "beta"}

# No es extensión de dominio: es inferencia de dominio desde valores observados.
assert domains_extended_bootstrap_ok == []

# No hubo value_correspondence.
assert vc_applied_bootstrap_ok == {}
assert n_map_bootstrap_ok == 0

# El schema efectivo debe registrar que el dominio fue inferido.
assert "bootstrap_cat" in schema_effective_bootstrap_ok.domains_effective
assert schema_effective_bootstrap_ok.dtype_effective["bootstrap_cat"] == "categorical"
assert "bootstrap_cat" in schema_effective_bootstrap_ok.overrides
assert (
    "categorical_domain_inferred_from_observed_values"
    in schema_effective_bootstrap_ok.overrides["bootstrap_cat"]["reasons"]
)

# Issue esperado
assert_issue_present(issues_bootstrap_ok, "DOM.INFERENCE.APPLIED")

# En este caso no debería degradarse a string.
assert_issue_absent(issues_bootstrap_ok, "DOM.INFERENCE.DEGRADED_TO_STRING")

assert_json_safe(domains_eff_bootstrap_ok, "domains_effective_bootstrap_ok")
assert_json_safe(schema_effective_bootstrap_ok.to_dict(), "schema_effective_bootstrap_ok")

show_ok("_standardize_categorical_values - dominio vacío con inferencia aplicada")

OK - _standardize_categorical_values - dominio vacío con inferencia aplicada


In [262]:
# canonical value fuera de dominio en value_correspondence -> error esperado si strict=True

value_corr_bad_canonical = {
    "mode": {
        "auto": "plane",  # no pertenece al dominio base y mode no es extendable
    }
}

try:
    _standardize_categorical_values(
        df_g5.copy(deep=True),
        schema=schema_g5,
        schema_effective=TripSchemaEffective(
            dtype_effective=dict(schema_effective_g5.dtype_effective),
            overrides={},
            domains_effective={},
            temporal={},
            fields_effective=[],
        ),
        value_correspondence=value_corr_bad_canonical,
        options=ImportOptions(strict=True, strict_domains=False),
        target_schema_fields=target_schema_fields_all,
    )
    assert False, "Debió lanzar PylondrinaImportError por canonical value fuera de dominio"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "MAP.VALUES.UNKNOWN_CANONICAL_VALUE")

show_ok("_standardize_categorical_values - canonical fuera de dominio")

OK - _standardize_categorical_values - canonical fuera de dominio


In [263]:
# extendable=True + strict_domains=True -> error esperado si strict=True

df_strict_domains = pd.DataFrame({
    "purpose": ["work", "funeral", pd.NA],
})

schema_strict_domains = TripSchema(
    version="0.1.0",
    fields={
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=["work", "study"],
                extendable=True,
                aliases=None,
            ),
        ),
    },
    required=[],
    semantic_rules=None,
)

schema_effective_strict_domains = TripSchemaEffective(
    dtype_effective={"purpose": "categorical"},
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

try:
    _standardize_categorical_values(
        df_strict_domains.copy(deep=True),
        schema=schema_strict_domains,
        schema_effective=schema_effective_strict_domains,
        value_correspondence=None,
        options=ImportOptions(strict=True, strict_domains=True),
        target_schema_fields={"purpose"},
    )
    assert False, "Debió lanzar PylondrinaImportError por strict_domains=True con valor fuera de dominio"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "DOM.STRICT.OUT_OF_DOMAIN_ABORT")

show_ok("_standardize_categorical_values - strict_domains=True")

OK - _standardize_categorical_values - strict_domains=True


In [264]:
# extendable=True + strict_domains=True con strict=False -> error

try:
    work_strict_false, domains_eff_strict_false, vc_applied_strict_false, domains_extended_strict_false, n_map_strict_false, issues_strict_false = (
        _standardize_categorical_values(
            df_strict_domains.copy(deep=True),
            schema=schema_strict_domains,
            schema_effective=TripSchemaEffective(
                dtype_effective={"purpose": "categorical"},
                overrides={},
                domains_effective={},
                temporal={},
                fields_effective=[],
            ),
            value_correspondence=None,
            options=ImportOptions(strict=False, strict_domains=True),
            target_schema_fields={"purpose"},
        )
    )
    assert False, "Debió lanzar PylondrinaImportError por strict_domains=True con valor fuera de dominio"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "DOM.STRICT.OUT_OF_DOMAIN_ABORT")

show_ok("_standardize_categorical_values - strict_domains=True con strict=False")

OK - _standardize_categorical_values - strict_domains=True con strict=False


In [265]:
# Test de interacción con target_schema_fields para probar que si un campo categórico no está seleccionado en target_schema_fields, no se procesa ni se registra en domains_effective.

target_schema_fields_partial = {"mode"}  # purpose y bootstrap_cat quedan fuera

value_corr_target = {
    "mode": {"auto": "car"},
    "purpose": {"trabajo": "work"},
}

work_target, domains_eff_target, vc_applied_target, domains_extended_target, n_map_target, issues_target = (
    _standardize_categorical_values(
        df_g5.copy(deep=True),
        schema=schema_g5,
        schema_effective=TripSchemaEffective(
            dtype_effective=dict(schema_effective_g5.dtype_effective),
            overrides={},
            domains_effective={},
            temporal={},
            fields_effective=[],
        ),
        value_correspondence=value_corr_target,
        options=ImportOptions(strict=False, strict_domains=False),
        target_schema_fields=target_schema_fields_partial,
    )
)

# mode sí se procesa
assert work_target["mode"].tolist() == ["car", "bus", "unknown", pd.NA]
assert "mode" in domains_eff_target

# purpose NO se procesa porque no está en target_schema_fields
assert work_target["purpose"].tolist() == ["trabajo", "study", "funeral", pd.NA]
assert "purpose" not in domains_eff_target

# bootstrap_cat tampoco se procesa
assert work_target["bootstrap_cat"].tolist() == ["alpha", "beta", pd.NA, "alpha"]
assert "bootstrap_cat" not in domains_eff_target

# value_correspondence_applied solo para mode
assert vc_applied_target == {"mode": {"auto": "car"}}
assert n_map_target == 1

show_ok("_standardize_categorical_values - target_schema_fields")

OK - _standardize_categorical_values - target_schema_fields


mode: 
- → campo categórico con dominio no extendible
- → valores fuera de dominio se mapean a unknown
- → reason: out_of_domain_mapped_to_unknown

purpose:
- → campo categórico con dominio extendible
- → valor nuevo se agrega al dominio efectivo
- → reason: domain_extended

bootstrap_cat:
- → campo categórico declarado con dominio vacío
- → intenta inferencia controlada
- → con pocos datos se degrada a string
- → reason: categorical_inference_degraded_to_string_high_cardinality

In [266]:
# actualización de schema_effective
# Actualizado para la política vigente de DomainSpec(values=[]):
# bootstrap_cat ya no se interpreta como extensión automática de dominio.
# Con pocos datos, la inferencia categórica se degrada a string.

schema_effective_mut = TripSchemaEffective(
    dtype_effective=dict(schema_effective_g5.dtype_effective),
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

work_mut, domains_eff_mut, vc_applied_mut, domains_extended_mut, n_map_mut, issues_mut = (
    _standardize_categorical_values(
        df_g5.copy(deep=True),
        schema=schema_g5,
        schema_effective=schema_effective_mut,
        value_correspondence={
            "mode": {"auto": "car"},
            "purpose": {"trabajo": "work"},
        },
        options=ImportOptions(strict=False, strict_domains=False),
        target_schema_fields=target_schema_fields_all,
    )
)

# domains_effective también debe quedar reflejado en schema_effective.
# Ojo: bootstrap_cat no debe estar, porque fue degradado a string.
assert set(schema_effective_mut.domains_effective.keys()) == set(domains_eff_mut.keys())
assert "bootstrap_cat" not in schema_effective_mut.domains_effective
assert "bootstrap_cat" not in domains_eff_mut

# overrides esperados por cambios importantes
assert "mode" in schema_effective_mut.overrides
assert "purpose" in schema_effective_mut.overrides
assert "bootstrap_cat" in schema_effective_mut.overrides

# mode: valor fuera de dominio no extendible -> unknown
assert "out_of_domain_mapped_to_unknown" in schema_effective_mut.overrides["mode"]["reasons"]

# purpose: dominio extendible, non-strict -> extensión aplicada
assert "domain_extended" in schema_effective_mut.overrides["purpose"]["reasons"]

# bootstrap_cat: dominio vacío, pero con pocos datos para inferencia -> degradación a string
assert (
    "categorical_inference_degraded_to_string_high_cardinality"
    in schema_effective_mut.overrides["bootstrap_cat"]["reasons"]
)
assert schema_effective_mut.dtype_effective["bootstrap_cat"] == "string"
assert schema_effective_mut.overrides["bootstrap_cat"]["fallback_dtype"] == "string"
assert schema_effective_mut.overrides["bootstrap_cat"]["observed_values_total"] == 2

# issues esperados
assert_issue_present(issues_mut, "DOM.POLICY.FIELD_NOT_EXTENDABLE")
assert_issue_present(issues_mut, "DOM.EXTENSION.APPLIED")
assert_issue_present(issues_mut, "DOM.INFERENCE.DEGRADED_TO_STRING")

assert_json_safe(schema_effective_mut.to_dict(), "schema_effective_mut")

show_ok("_standardize_categorical_values - schema_effective actualizado")

OK - _standardize_categorical_values - schema_effective actualizado


In [267]:
show_ok("Grupo 5 completo")

OK - Grupo 5 completo


## Sección 4. Tiempo, coerción, coordenadas y H3
### Grupo 2B - normalización temporal

En esta subsección se prueban las helpers de normalización temporal:

- `_normalize_datetime_column`
- `_normalize_datetime_columns`
- `_normalize_tier2_hhmm_columns`

Objetivo:
- comprobar que Tier 1 normaliza datetimes según política,
- que Tier 2 limpia HH:MM inválidos,
- y que los issues emitidos sean coherentes con el comportamiento esperado.

In [268]:
# Tests de _normalize_datetime_column

# Caso 1: datetime tz-aware UTC explícito
s_utc = pd.Series(pd.to_datetime(["2026-03-01T08:00:00Z", None], utc=True))
out_utc, info_utc = _normalize_datetime_column(s_utc, source_timezone=None)

assert str(out_utc.dtype) == "datetime64[ns, UTC]"
assert info_utc["status"] == "utc"
assert info_utc["n_nat"] == 1

# Caso 2: datetime naive + source_timezone
s_naive = pd.Series(pd.to_datetime(["2026-03-01 08:00:00", "2026-03-01 09:30:00"]))
out_naive, info_naive = _normalize_datetime_column(s_naive, source_timezone="America/Santiago")

assert str(out_naive.dtype) == "datetime64[ns, UTC]"
assert info_naive["status"] == "naive_localized_to_utc"
assert info_naive["tz_kind"] == "iana"

# Caso 3: string ISO con Z
s_str_utc = pd.Series(["2026-03-01T08:00:00Z", "2026-03-01T09:40:00Z"], dtype="object")
out_str_utc, info_str_utc = _normalize_datetime_column(s_str_utc, source_timezone=None)

assert str(out_str_utc.dtype) == "datetime64[ns, UTC]"
assert info_str_utc["status"] == "string_tzaware_to_utc"
assert info_str_utc["n_nat"] == 0

# Caso 4: string naive sin source_timezone
s_str_naive = pd.Series(["2026-03-01 08:00:00", "2026-03-01 09:40:00"], dtype="object")
out_str_naive, info_str_naive = _normalize_datetime_column(s_str_naive, source_timezone=None)

assert "datetime64[ns]" in str(out_str_naive.dtype)
assert info_str_naive["status"] == "string_naive_unconverted"

# Caso 5: numérico no parseable
s_num = pd.Series([1700000000, 1700003600], dtype="int64")
out_num, info_num = _normalize_datetime_column(s_num, source_timezone="UTC")

assert "datetime64[ns]" in str(out_num.dtype)
assert int(out_num.isna().sum()) == 2
assert info_num["status"] == "not_parsed_numeric"
assert info_num["n_nat"] == 2

show_ok("_normalize_datetime_column")

OK - _normalize_datetime_column


In [269]:
# Tests de _normalize_datetime_columns

schema_dt = TripSchema(
    version="0.1.0",
    fields={
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
        "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime", required=False),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
    },
    required=[],
    semantic_rules=None,
)

schema_effective_dt = TripSchemaEffective(
    dtype_effective={
        "origin_time_utc": "datetime",
        "destination_time_utc": "datetime",
        "trip_id": "string",
    }
)

# Caso 1: Tier 1 con strings UTC explícitos
df_dt_utc = pd.DataFrame({
    "origin_time_utc": ["2026-03-01T08:00:00Z", "2026-03-01T09:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z", "2026-03-01T09:40:00Z"],
    "trip_id": ["t1", "t2"],
})

work_dt_utc, status_dt_utc, issues_dt_utc = _normalize_datetime_columns(
    df_dt_utc.copy(deep=True),
    schema=schema_dt,
    schema_effective=schema_effective_dt,
    options=ImportOptions(source_timezone=None),
    temporal_tier="tier_1",
    strict=False,
)

assert str(work_dt_utc["origin_time_utc"].dtype) == "datetime64[ns, UTC]"
assert str(work_dt_utc["destination_time_utc"].dtype) == "datetime64[ns, UTC]"
assert status_dt_utc["origin_time_utc"]["status"] == "string_tzaware_to_utc"
assert status_dt_utc["destination_time_utc"]["status"] == "string_tzaware_to_utc"
assert_issue_absent(issues_dt_utc, "IMP.DATETIME.NUMERIC_NOT_PARSED")
assert_issue_absent(issues_dt_utc, "IMP.DATETIME.NAIVE_WITHOUT_SOURCE_TZ")

# Caso 2: Tier 1 con strings naive + source_timezone
df_dt_naive = pd.DataFrame({
    "origin_time_utc": ["2026-03-01 08:00:00", "2026-03-01 09:00:00"],
    "destination_time_utc": ["2026-03-01 08:30:00", "2026-03-01 09:40:00"],
})

work_dt_naive, status_dt_naive, issues_dt_naive = _normalize_datetime_columns(
    df_dt_naive.copy(deep=True),
    schema=schema_dt,
    schema_effective=schema_effective_dt,
    options=ImportOptions(source_timezone="America/Santiago"),
    temporal_tier="tier_1",
    strict=False,
)

assert str(work_dt_naive["origin_time_utc"].dtype) == "datetime64[ns, UTC]"
assert str(work_dt_naive["destination_time_utc"].dtype) == "datetime64[ns, UTC]"
assert status_dt_naive["origin_time_utc"]["status"] == "string_naive_localized_to_utc"
assert status_dt_naive["destination_time_utc"]["status"] == "string_naive_localized_to_utc"
assert_issue_absent(issues_dt_naive, "IMP.DATETIME.NAIVE_WITHOUT_SOURCE_TZ")

# Caso 3: Tier 1 numérico no parseable
df_dt_num = pd.DataFrame({
    "origin_time_utc": [1700000000, 1700003600],
    "destination_time_utc": [1700001800, 1700005400],
})

work_dt_num, status_dt_num, issues_dt_num = _normalize_datetime_columns(
    df_dt_num.copy(deep=True),
    schema=schema_dt,
    schema_effective=schema_effective_dt,
    options=ImportOptions(source_timezone="UTC"),
    temporal_tier="tier_1",
    strict=False,
)

assert int(work_dt_num["origin_time_utc"].isna().sum()) == 2
assert int(work_dt_num["destination_time_utc"].isna().sum()) == 2
assert status_dt_num["origin_time_utc"]["status"] == "not_parsed_numeric"
assert status_dt_num["destination_time_utc"]["status"] == "not_parsed_numeric"
assert_issue_present(issues_dt_num, "IMP.DATETIME.NUMERIC_NOT_PARSED")

# Caso 4: temporal_tier != tier_1 -> no toca nada
df_tier2_passthrough = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00", "09:00"],
    "destination_time_local_hhmm": ["08:30", "09:40"],
})

work_passthrough, status_passthrough, issues_passthrough = _normalize_datetime_columns(
    df_tier2_passthrough.copy(deep=True),
    schema=schema_dt,
    schema_effective=schema_effective_dt,
    options=ImportOptions(source_timezone=None),
    temporal_tier="tier_2",
    strict=False,
)

assert_columns_equal(work_passthrough, ["origin_time_local_hhmm", "destination_time_local_hhmm"], "tier2 passthrough columns")
assert status_passthrough == {}
assert issues_passthrough == []

show_ok("_normalize_datetime_columns")

OK - _normalize_datetime_columns


In [270]:
# Tests de _normalize_tier2_hhmm_columns
# Actualizado para la firma vigente:
# _normalize_tier2_hhmm_columns(work, *, temporal_tier, schema)
# Ya no recibe issues=[]; retorna local_issues.

schema_hhmm = TripSchema(
    version="0.1.0",
    fields={
        "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string", required=False),
        "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string", required=False),
    },
    required=[],
    semantic_rules=None,
)

# Caso 1: Tier 2 limpio
df_hhmm_clean = pd.DataFrame({
    "origin_time_local_hhmm": ["08:23", "12:30", "23:59"],
    "destination_time_local_hhmm": ["09:10", "13:00", "00:00"],
})

work_hhmm_clean, stats_hhmm_clean, issues_hhmm_clean = _normalize_tier2_hhmm_columns(
    df_hhmm_clean.copy(deep=True),
    temporal_tier="tier_2",
    schema=schema_hhmm,
)

assert_dtype(work_hhmm_clean, "origin_time_local_hhmm", "string")
assert_dtype(work_hhmm_clean, "destination_time_local_hhmm", "string")

assert work_hhmm_clean["origin_time_local_hhmm"].tolist() == ["08:23", "12:30", "23:59"]
assert work_hhmm_clean["destination_time_local_hhmm"].tolist() == ["09:10", "13:00", "00:00"]

assert stats_hhmm_clean["origin_time_local_hhmm"]["n_total"] == 3
assert stats_hhmm_clean["origin_time_local_hhmm"]["n_invalid"] == 0
assert stats_hhmm_clean["destination_time_local_hhmm"]["n_total"] == 3
assert stats_hhmm_clean["destination_time_local_hhmm"]["n_invalid"] == 0

assert issues_hhmm_clean == []


# Caso 2: Tier 2 con valores inválidos
df_hhmm_dirty = pd.DataFrame({
    "origin_time_local_hhmm": ["08:23", "24:00", "ab:cd", "", None, "07:05"],
    "destination_time_local_hhmm": ["09:10", "13:00", "99:99", "  ", np.nan, "00:00"],
})

work_hhmm_dirty, stats_hhmm_dirty, issues_hhmm_dirty = _normalize_tier2_hhmm_columns(
    df_hhmm_dirty.copy(deep=True),
    temporal_tier="tier_2",
    schema=schema_hhmm,
)

assert_dtype(work_hhmm_dirty, "origin_time_local_hhmm", "string")
assert_dtype(work_hhmm_dirty, "destination_time_local_hhmm", "string")

# origin:
# 24:00 y ab:cd inválidos;
# "", None -> NA como faltantes;
# 07:05 válido.
assert work_hhmm_dirty.loc[0, "origin_time_local_hhmm"] == "08:23"
assert pd.isna(work_hhmm_dirty.loc[1, "origin_time_local_hhmm"])
assert pd.isna(work_hhmm_dirty.loc[2, "origin_time_local_hhmm"])
assert pd.isna(work_hhmm_dirty.loc[3, "origin_time_local_hhmm"])
assert pd.isna(work_hhmm_dirty.loc[4, "origin_time_local_hhmm"])
assert work_hhmm_dirty.loc[5, "origin_time_local_hhmm"] == "07:05"

# destination:
# 99:99 inválido;
# "  ", np.nan -> NA como faltantes;
# 00:00 válido.
assert work_hhmm_dirty.loc[0, "destination_time_local_hhmm"] == "09:10"
assert work_hhmm_dirty.loc[1, "destination_time_local_hhmm"] == "13:00"
assert pd.isna(work_hhmm_dirty.loc[2, "destination_time_local_hhmm"])
assert pd.isna(work_hhmm_dirty.loc[3, "destination_time_local_hhmm"])
assert pd.isna(work_hhmm_dirty.loc[4, "destination_time_local_hhmm"])
assert work_hhmm_dirty.loc[5, "destination_time_local_hhmm"] == "00:00"

assert stats_hhmm_dirty["origin_time_local_hhmm"]["n_total"] == 6
assert stats_hhmm_dirty["origin_time_local_hhmm"]["n_invalid"] == 2

assert stats_hhmm_dirty["destination_time_local_hhmm"]["n_total"] == 6
assert stats_hhmm_dirty["destination_time_local_hhmm"]["n_invalid"] == 1

# Debe emitir IMP.TYPE.COERCE_PARTIAL cuando hubo inválidos.
# En este caso debería aparecer una vez por cada columna con inválidos:
# origin_time_local_hhmm y destination_time_local_hhmm.
assert_issue_present(issues_hhmm_dirty, "IMP.TYPE.COERCE_PARTIAL")
assert sum(issue.code == "IMP.TYPE.COERCE_PARTIAL" for issue in issues_hhmm_dirty) == 2


# Caso 3: temporal_tier != tier_2 -> passthrough
df_hhmm_passthrough = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00"],
    "destination_time_local_hhmm": ["08:30"],
})

work_hhmm_passthrough, stats_hhmm_passthrough, issues_hhmm_passthrough = _normalize_tier2_hhmm_columns(
    df_hhmm_passthrough.copy(deep=True),
    temporal_tier="tier_3",
    schema=schema_hhmm,
)

assert work_hhmm_passthrough.equals(df_hhmm_passthrough)
assert stats_hhmm_passthrough == {}
assert issues_hhmm_passthrough == []

show_ok("_normalize_tier2_hhmm_columns")

OK - _normalize_tier2_hhmm_columns


### Grupo 6 - coerción y parseo mínimo

En esta sección se prueban helpers que hacen coerción mínima de tipos y parseo básico de coordenadas.

Objetivo:
- verificar que `import` transforme lo mínimo prometido,
- comprobar coerción parcial con valores sucios,
- distinguir cuándo un campo requerido queda inutilizable,
- y asegurar que el parseo de coordenadas soporte DD, DM y DMS, dejando inválidos como `NaN`.

In [271]:
# Tests de _coerce_series_to_dtype

# Caso 1: string
s_str = pd.Series(["  a  ", "", None, "b"], dtype="object")
out_str, stats_str = _coerce_series_to_dtype(s_str, "string")

assert str(out_str.dtype) == "string"
assert out_str.tolist() == ["a", pd.NA, pd.NA, "b"]
assert stats_str["expected"] == "string"
assert stats_str["dtype_before"] == "object"
assert stats_str["dtype_after"] == "string"
assert stats_str["na_before"] == 1
assert stats_str["na_after"] == 2
assert stats_str["na_delta"] == 1

# Caso 2: int con basura
s_int = pd.Series(["1", "2", "abc", None], dtype="object")
out_int, stats_int = _coerce_series_to_dtype(s_int, "int")

assert str(out_int.dtype) == "Int64"
assert out_int.tolist() == [1, 2, pd.NA, pd.NA]
assert stats_int["na_delta"] == 1

# Caso 3: float con basura
s_float = pd.Series(["1.5", "2", "abc", None], dtype="object")
out_float, stats_float = _coerce_series_to_dtype(s_float, "float")

assert str(out_float.dtype) == "float64"
assert math.isclose(out_float.iloc[0], 1.5, rel_tol=0, abs_tol=1e-12)
assert math.isclose(out_float.iloc[1], 2.0, rel_tol=0, abs_tol=1e-12)
assert pd.isna(out_float.iloc[2])
assert pd.isna(out_float.iloc[3])
assert stats_float["na_delta"] == 1

# Caso 4: bool nullable
s_bool = pd.Series(["true", "FALSE", "1", "0", "yes", "no", "abc", "", None], dtype="object")
out_bool, stats_bool = _coerce_series_to_dtype(s_bool, "bool")

assert str(out_bool.dtype) == "boolean"
assert out_bool.tolist() == [True, False, True, False, True, False, pd.NA, pd.NA, pd.NA]
assert stats_bool["na_before"] == 1
assert stats_bool["na_after"] == 3
assert stats_bool["na_delta"] == 2

# Caso 5: datetime con parse_datetime=False -> no toca strings
s_dt_no_parse = pd.Series(["2026-03-01 08:00:00", "2026-03-01 09:00:00"], dtype="object")
out_dt_no_parse, stats_dt_no_parse = _coerce_series_to_dtype(s_dt_no_parse, "datetime", parse_datetime=False)

assert out_dt_no_parse.equals(s_dt_no_parse)
assert stats_dt_no_parse["dtype_before"] == "object"
assert stats_dt_no_parse["dtype_after"] == "object"

# Caso 6: datetime con parse_datetime=True
s_dt_parse = pd.Series(["2026-03-01 08:00:00", "bad", None], dtype="object")
out_dt_parse, stats_dt_parse = _coerce_series_to_dtype(s_dt_parse, "datetime", parse_datetime=True)

assert "datetime64[ns]" in str(out_dt_parse.dtype)
assert pd.notna(out_dt_parse.iloc[0])
assert pd.isna(out_dt_parse.iloc[1])
assert pd.isna(out_dt_parse.iloc[2])
assert stats_dt_parse["na_delta"] == 1

show_ok("_coerce_series_to_dtype")

OK - _coerce_series_to_dtype


In [272]:
# Fixtures comunes para _coerce_columns_by_dtype

schema_g6 = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "trip_count": FieldSpec(name="trip_count", dtype="int", required=False),
        "distance_km": FieldSpec(name="distance_km", dtype="float", required=False),
        "is_student": FieldSpec(name="is_student", dtype="bool", required=False),
        "required_int": FieldSpec(name="required_int", dtype="int", required=True),
    },
    required=["user_id", "required_int"],
    semantic_rules=None,
)

schema_effective_g6 = TripSchemaEffective(
    dtype_effective={
        "user_id": "string",
        "trip_count": "int",
        "distance_km": "float",
        "is_student": "bool",
        "required_int": "int",
    },
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

target_fields_g6 = {"user_id", "trip_count", "distance_km", "is_student", "required_int"}

In [273]:
# Tests de _coerce_columns_by_dtype -> coerción parcial

df_coerce_partial = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "trip_count": ["1", "abc", "3"],
    "distance_km": ["1.5", "2.0", "bad"],
    "is_student": ["true", "no", "abc"],
    "required_int": ["10", "20", "30"],
})

work_coerce_partial, coercion_stats_partial, issues_coerce_partial = _coerce_columns_by_dtype(
    df_coerce_partial.copy(deep=True),
    schema=schema_g6,
    schema_effective=schema_effective_g6,
    target_schema_fields=target_fields_g6,
    strict=False,
)

assert_dtype(work_coerce_partial, "user_id", "string")
assert_dtype(work_coerce_partial, "trip_count", "Int64")
assert_dtype(work_coerce_partial, "distance_km", "float64")
assert_dtype(work_coerce_partial, "is_student", "boolean")
assert_dtype(work_coerce_partial, "required_int", "Int64")

assert work_coerce_partial["trip_count"].tolist() == [1, pd.NA, 3]
assert pd.isna(work_coerce_partial.loc[2, "distance_km"])
assert work_coerce_partial["is_student"].tolist() == [True, False, pd.NA]

assert coercion_stats_partial["trip_count"]["na_delta"] == 1
assert coercion_stats_partial["distance_km"]["na_delta"] == 1
assert coercion_stats_partial["is_student"]["na_delta"] == 1

assert_issue_present(issues_coerce_partial, "IMP.TYPE.COERCE_PARTIAL")

show_ok("_coerce_columns_by_dtype - coerción parcial")

OK - _coerce_columns_by_dtype - coerción parcial


In [274]:
# _coerce_columns_by_dtype -> required crítico inutilizable

df_coerce_required_fail = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "trip_count": ["1", "2", "3"],
    "distance_km": ["1.5", "2.0", "3.1"],
    "is_student": ["true", "false", "true"],
    "required_int": ["bad", "worse", "nope"],
})

try:
    _coerce_columns_by_dtype(
        df_coerce_required_fail.copy(deep=True),
        schema=schema_g6,
        schema_effective=schema_effective_g6,
        target_schema_fields=target_fields_g6,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por required_int inutilizable"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.TYPE.COERCE_FAILED_REQUIRED")

show_ok("_coerce_columns_by_dtype - required crítico inutilizable")

OK - _coerce_columns_by_dtype - required crítico inutilizable


In [275]:
# Fixtures comunes para _parse_od_coordinate_columns

schema_coords_optional = TripSchema(
    version="0.1.0",
    fields={
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
    },
    required=[],
    semantic_rules=None,
)

schema_coords_required = TripSchema(
    version="0.1.0",
    fields={
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
    },
    required=["origin_latitude"],
    semantic_rules=None,
)

target_fields_coords = {
    "origin_latitude",
    "origin_longitude",
    "destination_latitude",
    "destination_longitude",
}

In [276]:
# Tests de _parse_od_coordinate_columns -> DD, DM, DMS e inválidos

df_coords_mixed = pd.DataFrame({
    "origin_latitude": ["-33.446160", "33 27.0000 S", "33 27 00 S", "abc"],
    "origin_longitude": ["-70.572755", "70 34.3653 W", "70 30 00 W", "xyz"],
    "destination_latitude": ["-33.392693", pd.NA, "33 20 00 S", ""],
    "destination_longitude": ["-70.517930", None, "70 31 00 W", "  "],
})

work_coords_mixed, coord_stats_mixed, issues_coords_mixed = _parse_od_coordinate_columns(
    df_coords_mixed.copy(deep=True),
    schema=schema_coords_optional,
    target_schema_fields=target_fields_coords,
    strict=False,
)

assert_dtype(work_coords_mixed, "origin_latitude", "float64")
assert_dtype(work_coords_mixed, "origin_longitude", "float64")
assert_dtype(work_coords_mixed, "destination_latitude", "float64")
assert_dtype(work_coords_mixed, "destination_longitude", "float64")

# DD
assert math.isclose(work_coords_mixed.loc[0, "origin_latitude"], -33.446160, rel_tol=0, abs_tol=1e-9)
assert math.isclose(work_coords_mixed.loc[0, "origin_longitude"], -70.572755, rel_tol=0, abs_tol=1e-9)

# DM
assert math.isclose(work_coords_mixed.loc[1, "origin_latitude"], -33.45, rel_tol=0, abs_tol=1e-9)
assert math.isclose(work_coords_mixed.loc[1, "origin_longitude"], -(70 + 34.3653/60.0), rel_tol=0, abs_tol=1e-9)

# DMS
assert math.isclose(work_coords_mixed.loc[2, "origin_latitude"], -33.45, rel_tol=0, abs_tol=1e-9)
assert math.isclose(work_coords_mixed.loc[2, "origin_longitude"], -70.5, rel_tol=0, abs_tol=1e-9)

# inválidos -> NaN
assert pd.isna(work_coords_mixed.loc[3, "origin_latitude"])
assert pd.isna(work_coords_mixed.loc[3, "origin_longitude"])
assert pd.isna(work_coords_mixed.loc[3, "destination_latitude"])
assert pd.isna(work_coords_mixed.loc[3, "destination_longitude"])

# stats deben existir
assert "origin_latitude" in coord_stats_mixed
assert "origin_longitude" in coord_stats_mixed
assert coord_stats_mixed["origin_latitude"]["parse_fail_count"] == 1
assert coord_stats_mixed["origin_longitude"]["parse_fail_count"] == 1

assert_issue_present(issues_coords_mixed, "IMP.TYPE.COERCE_PARTIAL")

show_ok("_parse_od_coordinate_columns - DD, DM, DMS e inválidos")

OK - _parse_od_coordinate_columns - DD, DM, DMS e inválidos


In [277]:
# Tests de _parse_od_coordinate_columns -> required crítico inutilizable

df_coords_required_fail = pd.DataFrame({
    "origin_latitude": ["abc", "def", "ghi"],
    "origin_longitude": ["-70.6", "-70.7", "-70.8"],
})

try:
    _parse_od_coordinate_columns(
        df_coords_required_fail.copy(deep=True),
        schema=schema_coords_required,
        target_schema_fields={"origin_latitude", "origin_longitude"},
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por origin_latitude requerida e inutilizable"
except PylondrinaImportError as error:
    print("Mensaje del error:", error.message)
    assert_issue_present(error.issues, "IMP.TYPE.COERCE_FAILED_REQUIRED")

show_ok("_parse_od_coordinate_columns - required crítico inutilizable")

Mensaje del error: No fue posible convertir el campo requerido 'origin_latitude' a un formato mínimo utilizable; import abortado.
OK - _parse_od_coordinate_columns - required crítico inutilizable


In [278]:
# Tests de _parse_od_coordinate_columns -> target_schema_fields limita el trabajo

df_coords_target = pd.DataFrame({
    "origin_latitude": ["33 27 00 S"],
    "origin_longitude": ["70 30 00 W"],
    "destination_latitude": ["abc"],
    "destination_longitude": ["xyz"],
})

work_coords_target, coord_stats_target, issues_coords_target = _parse_od_coordinate_columns(
    df_coords_target.copy(deep=True),
    schema=schema_coords_optional,
    target_schema_fields={"origin_latitude", "origin_longitude"},
    strict=False,
)

# origin se procesa
assert math.isclose(work_coords_target.loc[0, "origin_latitude"], -33.45, rel_tol=0, abs_tol=1e-9)
assert math.isclose(work_coords_target.loc[0, "origin_longitude"], -70.5, rel_tol=0, abs_tol=1e-9)

# destination no se toca porque no está en target_schema_fields
assert work_coords_target.loc[0, "destination_latitude"] == "abc"
assert work_coords_target.loc[0, "destination_longitude"] == "xyz"

assert set(coord_stats_target.keys()) == {"origin_latitude", "origin_longitude"}
assert issues_coords_target == []

show_ok("_parse_od_coordinate_columns - target_schema_fields")

OK - _parse_od_coordinate_columns - target_schema_fields


In [279]:
# Test integrado pequeño del Grupo 6: coerción + coordenadas

schema_g6_integrated = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "trip_count": FieldSpec(name="trip_count", dtype="int", required=False),
        "is_student": FieldSpec(name="is_student", dtype="bool", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

schema_effective_g6_integrated = TripSchemaEffective(
    dtype_effective={
        "user_id": "string",
        "trip_count": "int",
        "is_student": "bool",
        "origin_latitude": "float",
        "origin_longitude": "float",
    },
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[],
)

df_g6_integrated = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "trip_count": ["1", "bad", "3"],
    "is_student": ["true", "no", "abc"],
    "origin_latitude": ["-33.446160", "33 27 00 S", "abc"],
    "origin_longitude": ["-70.572755", "70 30 00 W", "xyz"],
})

work_g6_integrated, coercion_stats_integrated, issues_coercion_integrated = _coerce_columns_by_dtype(
    df_g6_integrated.copy(deep=True),
    schema=schema_g6_integrated,
    schema_effective=schema_effective_g6_integrated,
    target_schema_fields={"user_id", "trip_count", "is_student", "origin_latitude", "origin_longitude"},
    strict=False,
)

work_g6_integrated_2, coord_stats_integrated, issues_coords_integrated = _parse_od_coordinate_columns(
    work_g6_integrated,
    schema=schema_g6_integrated,
    target_schema_fields={"origin_latitude", "origin_longitude"},
    strict=False,
)

assert_dtype(work_g6_integrated_2, "trip_count", "Int64")
assert_dtype(work_g6_integrated_2, "is_student", "boolean")
assert_dtype(work_g6_integrated_2, "origin_latitude", "float64")
assert_dtype(work_g6_integrated_2, "origin_longitude", "float64")

assert work_g6_integrated_2["trip_count"].tolist() == [1, pd.NA, 3]
assert work_g6_integrated_2["is_student"].tolist() == [True, False, pd.NA]
calc = work_g6_integrated_2.loc[1, "origin_latitude"]
assert math.isclose(calc, -33.45, rel_tol=0, abs_tol=1e-9), "En verdad calc es: " + str(calc)
assert math.isclose(work_g6_integrated_2.loc[1, "origin_longitude"], -70.5, rel_tol=0, abs_tol=1e-9)
assert pd.isna(work_g6_integrated_2.loc[2, "origin_latitude"])
assert pd.isna(work_g6_integrated_2.loc[2, "origin_longitude"])

assert_issue_present(issues_coercion_integrated, "IMP.TYPE.COERCE_PARTIAL")
assert_issue_present(issues_coords_integrated, "IMP.TYPE.COERCE_PARTIAL")

show_ok("Grupo 6 - test integrado pequeño")

OK - Grupo 6 - test integrado pequeño


In [280]:
show_ok("Grupo 6 completo")

OK - Grupo 6 completo


### Grupo 7 - derivaciones espaciales

En esta sección se prueba la helper `_derive_h3_indices(...)`.

Objetivo:
- verificar que derive índices H3 cuando hay coordenadas utilizables,
- comprobar el comportamiento con coordenadas parcialmente nulas,
- revisar qué pasa cuando los H3 son requeridos pero no se pueden materializar,
- y dejar congelado el comportamiento actual ante resolución inválida.

In [281]:
# Fixtures comunes del Grupo 7

schema_h3_optional = TripSchema(
    version="0.1.0",
    fields={
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=False),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=False),
    },
    required=[],
    semantic_rules=None,
)

schema_h3_required_origin = TripSchema(
    version="0.1.0",
    fields={
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=True),
    },
    required=["origin_h3_index"],
    semantic_rules=None,
)

schema_h3_required_destination = TripSchema(
    version="0.1.0",
    fields={
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=True),
    },
    required=["destination_h3_index"],
    semantic_rules=None,
)

In [282]:
# Tests de _derive_h3_indices -> con lat/lon completos

df_h3_full = pd.DataFrame({
    "origin_latitude": [-33.446160, -33.392693],
    "origin_longitude": [-70.572755, -70.517930],
    "destination_latitude": [-33.447000, -33.390000],
    "destination_longitude": [-70.570000, -70.510000],
})

work_h3_full, h3_meta_full, columns_added_full, issues_h3_full = _derive_h3_indices(
    df_h3_full.copy(deep=True),
    schema=schema_h3_optional,
    h3_resolution=8,
    strict=False,
)

assert "origin_h3_index" in work_h3_full.columns
assert "destination_h3_index" in work_h3_full.columns

assert_dtype(work_h3_full, "origin_h3_index", "string")
assert_dtype(work_h3_full, "destination_h3_index", "string")

expected_o0 = h3.latlng_to_cell(-33.446160, -70.572755, 8)
expected_o1 = h3.latlng_to_cell(-33.392693, -70.517930, 8)
expected_d0 = h3.latlng_to_cell(-33.447000, -70.570000, 8)
expected_d1 = h3.latlng_to_cell(-33.390000, -70.510000, 8)

assert work_h3_full["origin_h3_index"].tolist() == [expected_o0, expected_o1]
assert work_h3_full["destination_h3_index"].tolist() == [expected_d0, expected_d1]

assert columns_added_full == ["origin_h3_index", "destination_h3_index"]
assert h3_meta_full["resolution"] == 8
assert h3_meta_full["source_fields"] == [
    ["origin_latitude", "origin_longitude"],
    ["destination_latitude", "destination_longitude"],
]
assert h3_meta_full["derived_fields"] == ["origin_h3_index", "destination_h3_index"]

assert issues_h3_full == []

show_ok("_derive_h3_indices - lat/lon completos")

OK - _derive_h3_indices - lat/lon completos


In [283]:
# Tests de _derive_h3_indices -> lat/lon parcialmente nulos

df_h3_partial = pd.DataFrame({
    "origin_latitude": [-33.446160, pd.NA, -33.392693],
    "origin_longitude": [-70.572755, -70.517930, pd.NA],
    "destination_latitude": [-33.447000, -33.390000, pd.NA],
    "destination_longitude": [-70.570000, pd.NA, -70.510000],
})

work_h3_partial, h3_meta_partial, columns_added_partial, issues_h3_partial = _derive_h3_indices(
    df_h3_partial.copy(deep=True),
    schema=schema_h3_optional,
    h3_resolution=8,
    strict=False,
)

assert "origin_h3_index" in work_h3_partial.columns
assert "destination_h3_index" in work_h3_partial.columns

assert_dtype(work_h3_partial, "origin_h3_index", "string")
assert_dtype(work_h3_partial, "destination_h3_index", "string")

# fila 0 válida
assert pd.notna(work_h3_partial.loc[0, "origin_h3_index"])
assert pd.notna(work_h3_partial.loc[0, "destination_h3_index"])

# filas parciales -> NA
assert pd.isna(work_h3_partial.loc[1, "origin_h3_index"])
assert pd.isna(work_h3_partial.loc[2, "origin_h3_index"])
assert pd.isna(work_h3_partial.loc[1, "destination_h3_index"])
assert pd.isna(work_h3_partial.loc[2, "destination_h3_index"])

assert columns_added_partial == ["origin_h3_index", "destination_h3_index"]
assert h3_meta_partial["derived_fields"] == ["origin_h3_index", "destination_h3_index"]

# Debe emitir derivación parcial
assert_issue_present(issues_h3_partial, "IMP.H3.PARTIAL_DERIVATION")

show_ok("_derive_h3_indices - lat/lon parcialmente nulos")

OK - _derive_h3_indices - lat/lon parcialmente nulos


In [284]:
# Tests de _derive_h3_indices -> H3 requerido pero no materializable

df_h3_missing_pair = pd.DataFrame({
    "origin_latitude": [-33.446160, -33.392693],
    # falta origin_longitude
})

try:
    _derive_h3_indices(
        df_h3_missing_pair.copy(deep=True),
        schema=schema_h3_required_origin,
        h3_resolution=8,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por H3 requerido no materializable"
except PylondrinaImportError as error:
    print("Mensaje del error: ", error.message)
    assert_issue_present(error.issues, "IMP.H3.REQUIRED_FIELDS_UNAVAILABLE")

show_ok("_derive_h3_indices - H3 requerido no materializable")

Mensaje del error:  No es posible materializar los índices H3 requeridos por el schema porque faltan campos fuente utilizables.
OK - _derive_h3_indices - H3 requerido no materializable


In [285]:
# Tests de _derive_h3_indices -> H3 opcional sin pares completos

df_h3_optional_missing_pair = pd.DataFrame({
    "origin_latitude": [-33.446160, -33.392693],
    # falta origin_longitude
    "destination_latitude": [-33.447000, -33.390000],
    # falta destination_longitude
})

work_h3_optional_missing, h3_meta_optional_missing, columns_added_optional_missing, issues_h3_optional_missing = _derive_h3_indices(
    df_h3_optional_missing_pair.copy(deep=True),
    schema=schema_h3_optional,
    h3_resolution=8,
    strict=False,
)

# No debe crear columnas si no tiene pares completos
assert_columns_equal(
    work_h3_optional_missing,
    ["origin_latitude", "destination_latitude"],
    "H3 opcional sin pares completos"
)

assert h3_meta_optional_missing == {}
assert columns_added_optional_missing == []
assert issues_h3_optional_missing == []

show_ok("_derive_h3_indices - H3 opcional sin pares completos")

OK - _derive_h3_indices - H3 opcional sin pares completos


In [286]:
#  Tests de _derive_h3_indices -> resolución inválida (comportamiento actual)

df_h3_bad_res = pd.DataFrame({
    "origin_latitude": [-33.446160, -33.392693],
    "origin_longitude": [-70.572755, -70.517930],
})

work_h3_bad_res, h3_meta_bad_res, columns_added_bad_res, issues_h3_bad_res = _derive_h3_indices(
    df_h3_bad_res.copy(deep=True),
    schema=schema_h3_optional,
    h3_resolution=99,   # inválida
    strict=False,
)

assert "origin_h3_index" in work_h3_bad_res.columns
assert_dtype(work_h3_bad_res, "origin_h3_index", "string")

# Como la resolución es inválida, las filas quedan en NA por el comportamiento actual
assert pd.isna(work_h3_bad_res.loc[0, "origin_h3_index"])
assert pd.isna(work_h3_bad_res.loc[1, "origin_h3_index"])

assert columns_added_bad_res == ["origin_h3_index"]
assert h3_meta_bad_res["derived_fields"] == ["origin_h3_index"]

# La helper actual emite derivación parcial, no invalid resolution
assert_issue_present(issues_h3_bad_res, "IMP.H3.PARTIAL_DERIVATION")

show_ok("_derive_h3_indices - resolución inválida (comportamiento actual)")

OK - _derive_h3_indices - resolución inválida (comportamiento actual)


In [287]:
# Test integrado pequeño del Grupo 7: origin + destination con una derivación parcial

df_h3_integrated = pd.DataFrame({
    "origin_latitude": [-33.446160, -33.392693, pd.NA],
    "origin_longitude": [-70.572755, -70.517930, -70.510000],
    "destination_latitude": [-33.447000, pd.NA, -33.390000],
    "destination_longitude": [-70.570000, -70.515000, -70.510000],
})

work_h3_integrated, h3_meta_integrated, columns_added_integrated, issues_h3_integrated = _derive_h3_indices(
    df_h3_integrated.copy(deep=True),
    schema=schema_h3_optional,
    h3_resolution=8,
    strict=False,
)

assert_columns_equal(
    work_h3_integrated,
    [
        "origin_latitude", "origin_longitude",
        "destination_latitude", "destination_longitude",
        "origin_h3_index", "destination_h3_index",
    ],
    "integrated H3 columns",
)

assert_dtype(work_h3_integrated, "origin_h3_index", "string")
assert_dtype(work_h3_integrated, "destination_h3_index", "string")

# primera fila completa
assert pd.notna(work_h3_integrated.loc[0, "origin_h3_index"])
assert pd.notna(work_h3_integrated.loc[0, "destination_h3_index"])

# filas con nulls parciales
assert pd.isna(work_h3_integrated.loc[2, "origin_h3_index"])
assert pd.isna(work_h3_integrated.loc[1, "destination_h3_index"])

assert columns_added_integrated == ["origin_h3_index", "destination_h3_index"]
assert h3_meta_integrated["resolution"] == 8
assert h3_meta_integrated["derived_fields"] == ["origin_h3_index", "destination_h3_index"]

assert_issue_present(issues_h3_integrated, "IMP.H3.PARTIAL_DERIVATION")

show_ok("Grupo 7 - test integrado pequeño")

OK - Grupo 7 - test integrado pequeño


In [288]:
show_ok("Grupo 7 completo")

OK - Grupo 7 completo


## Sección 5. IDs, selección final y required final
### Grupo 8 - IDs estructurales

En esta sección se prueban las helpers que garantizan la estructura mínima de IDs del dataset importado.

Objetivo:
- verificar unicidad mínima de `movement_id`,
- comprobar la generación automática de `movement_id` cuando falta,
- revisar el comportamiento de `single_stage=True` y `single_stage=False`,
- y asegurar que `trip_id` y `movement_seq` se creen solo cuando corresponde.

In [289]:
# Fixtures comunes del Grupo 8

schema_ids_required = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
    },
    required=["movement_id", "trip_id", "movement_seq"],
    semantic_rules=None,
)

schema_ids_not_required = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
    },
    required=["movement_id"],
    semantic_rules=None,
)

In [290]:
# Tests de _ensure_movement_id -> Con movement_id existente sin duplicados

df_mid_ok = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1", "m2"], dtype="string"),
    "purpose": ["work", "study", "home"],
})

work_mid_ok, columns_added_mid_ok, issues_mid_ok = _ensure_movement_id(
    df_mid_ok.copy(deep=True),
    strict=False,
)

assert_columns_equal(work_mid_ok, ["movement_id", "purpose"], "movement_id existente sin duplicados")
assert work_mid_ok["movement_id"].tolist() == ["m0", "m1", "m2"]
assert columns_added_mid_ok == []
assert issues_mid_ok == []

show_ok("_ensure_movement_id - existente sin duplicados")

OK - _ensure_movement_id - existente sin duplicados


In [291]:
# Tests de _ensure_movement_id -> con movement_id duplicado

df_mid_dup = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1", "m1", "m2"], dtype="string"),
    "purpose": ["work", "study", "home", "other"],
})

try:
    _ensure_movement_id(
        df_mid_dup.copy(deep=True),
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por movement_id duplicado"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.ID.MOVEMENT_ID_DUPLICATE")

show_ok("_ensure_movement_id - duplicado")

OK - _ensure_movement_id - duplicado


In [292]:
# Tests de _ensure_movement_id -> con movement_id faltante

df_mid_missing = pd.DataFrame({
    "purpose": ["work", "study", "home"],
    "mode": ["bus", "metro", "walk"],
})

work_mid_missing, columns_added_mid_missing, issues_mid_missing = _ensure_movement_id(
    df_mid_missing.copy(deep=True),
    strict=False,
)

assert_columns_equal(work_mid_missing, ["movement_id", "purpose", "mode"], "movement_id generado")
assert_dtype(work_mid_missing, "movement_id", "string")
assert work_mid_missing["movement_id"].tolist() == ["m0", "m1", "m2"]
assert columns_added_mid_missing == ["movement_id"]
assert_issue_present(issues_mid_missing, "IMP.ID.MOVEMENT_ID_CREATED")

show_ok("_ensure_movement_id - faltante")

OK - _ensure_movement_id - faltante


In [293]:
# Tests de _ensure_single_stage_ids -> single_stage=True genera trip_id y movement_seq

df_single_true_missing = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1", "m2"], dtype="string"),
    "purpose": ["work", "study", "home"],
})

work_single_true, columns_added_single_true, issues_single_true = _ensure_single_stage_ids(
    df_single_true_missing.copy(deep=True),
    schema=schema_ids_required,
    single_stage=True,
    strict=False,
)

assert_columns_equal(
    work_single_true,
    ["movement_id", "trip_id", "movement_seq", "purpose"],
    "single_stage=True genera IDs"
)
assert_dtype(work_single_true, "trip_id", "string")
assert_dtype(work_single_true, "movement_seq", "Int64")

assert work_single_true["trip_id"].tolist() == ["m0", "m1", "m2"]
assert work_single_true["movement_seq"].tolist() == [0, 0, 0]

assert columns_added_single_true == ["trip_id", "movement_seq"]
assert_issue_present(issues_single_true, "IMP.ID.TRIP_ID_CREATED")
assert_issue_present(issues_single_true, "IMP.ID.MOVEMENT_SEQ_CREATED")

show_ok("_ensure_single_stage_ids - single_stage=True generando campos")

OK - _ensure_single_stage_ids - single_stage=True generando campos


In [294]:
# Tests de _ensure_single_stage_ids -> single_stage=True no toca si ya existen

df_single_true_existing = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1"], dtype="string"),
    "trip_id": pd.Series(["t0", "t1"], dtype="string"),
    "movement_seq": pd.Series([0, 1], dtype="Int64"),
    "purpose": ["work", "study"],
})

work_single_true_existing, columns_added_single_true_existing, issues_single_true_existing = _ensure_single_stage_ids(
    df_single_true_existing.copy(deep=True),
    schema=schema_ids_required,
    single_stage=True,
    strict=False,
)

assert_columns_equal(
    work_single_true_existing,
    ["movement_id", "trip_id", "movement_seq", "purpose"],
    "single_stage=True con campos existentes"
)
assert work_single_true_existing["trip_id"].tolist() == ["t0", "t1"]
assert work_single_true_existing["movement_seq"].tolist() == [0, 1]

assert columns_added_single_true_existing == []
assert issues_single_true_existing == []

show_ok("_ensure_single_stage_ids - single_stage=True sin generar")

OK - _ensure_single_stage_ids - single_stage=True sin generar


In [295]:
# Tests de _ensure_single_stage_ids -> single_stage=True sin movement_id

df_single_true_no_mid = pd.DataFrame({
    "purpose": ["work", "study"],
})

try:
    _ensure_single_stage_ids(
        df_single_true_no_mid.copy(deep=True),
        schema=schema_ids_required,
        single_stage=True,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por faltar movement_id"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_ensure_single_stage_ids - single_stage=True sin movement_id")

OK - _ensure_single_stage_ids - single_stage=True sin movement_id


In [296]:
# _ensure_single_stage_ids -> single_stage=False sin generar campos

df_single_false_ok = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1"], dtype="string"),
    "purpose": ["work", "study"],
})

work_single_false_ok, columns_added_single_false_ok, issues_single_false_ok = _ensure_single_stage_ids(
    df_single_false_ok.copy(deep=True),
    schema=schema_ids_not_required,
    single_stage=False,
    strict=False,
)

assert_columns_equal(
    work_single_false_ok,
    ["movement_id", "purpose"],
    "single_stage=False no genera"
)
assert columns_added_single_false_ok == []
assert issues_single_false_ok == []

show_ok("_ensure_single_stage_ids - single_stage=False sin generar")

OK - _ensure_single_stage_ids - single_stage=False sin generar


In [297]:
# Tests de _ensure_single_stage_ids -> single_stage=False con schema que exige trip_id y movement_seq

df_single_false_missing_required = pd.DataFrame({
    "movement_id": pd.Series(["m0", "m1"], dtype="string"),
    "purpose": ["work", "study"],
})

try:
    _ensure_single_stage_ids(
        df_single_false_missing_required.copy(deep=True),
        schema=schema_ids_required,
        single_stage=False,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por faltar trip_id/movement_seq con single_stage=False"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_ensure_single_stage_ids - single_stage=False con required faltantes")

OK - _ensure_single_stage_ids - single_stage=False con required faltantes


In [298]:
# Test integrado pequeño del Grupo 8: ensure_movement_id + ensure_single_stage_ids

df_ids_integrated = pd.DataFrame({
    "purpose": ["work", "study", "home"],
})

work_ids_integrated, columns_added_mid_integrated, issues_mid_integrated = _ensure_movement_id(
    df_ids_integrated.copy(deep=True),
    strict=False,
)

work_ids_integrated_2, columns_added_stage_integrated, issues_stage_integrated = _ensure_single_stage_ids(
    work_ids_integrated,
    schema=schema_ids_required,
    single_stage=True,
    strict=False,
)

assert_columns_equal(
    work_ids_integrated_2,
    ["movement_id", "trip_id", "movement_seq", "purpose"],
    "integrated IDs"
)

assert work_ids_integrated_2["movement_id"].tolist() == ["m0", "m1", "m2"]
assert work_ids_integrated_2["trip_id"].tolist() == ["m0", "m1", "m2"]
assert work_ids_integrated_2["movement_seq"].tolist() == [0, 0, 0]

assert columns_added_mid_integrated == ["movement_id"]
assert columns_added_stage_integrated == ["trip_id", "movement_seq"]

assert_issue_present(issues_mid_integrated, "IMP.ID.MOVEMENT_ID_CREATED")
assert_issue_present(issues_stage_integrated, "IMP.ID.TRIP_ID_CREATED")
assert_issue_present(issues_stage_integrated, "IMP.ID.MOVEMENT_SEQ_CREATED")

show_ok("Grupo 8 - test integrado pequeño")

OK - Grupo 8 - test integrado pequeño


In [299]:
show_ok("Grupo 8 completo")

OK - Grupo 8 completo


### Grupo 9 - selección final y poda del schema efectivo

En esta sección se prueba la etapa final del dataframe antes de construir metadata y dataset.

Objetivo:
- verificar que la selección final de columnas respete required, selected_fields y keep_extra_fields,
- comprobar que el chequeo final de required detecte faltantes estructurales,
- y asegurar que `schema_effective` quede podado en coherencia con el dataframe final.

In [300]:
# Fixtures comunes del Grupo 9

schema_g9 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
    },
    required=["movement_id", "trip_id", "movement_seq"],
    semantic_rules=None,
)

df_g9 = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2"],
    "trip_id": ["t0", "t1", "t2"],
    "movement_seq": pd.Series([0, 0, 0], dtype="Int64"),
    "purpose": ["work", "study", "home"],
    "mode": ["bus", "metro", "walk"],
    "origin_latitude": [-33.45, -33.46, -33.47],
    "origin_longitude": [-70.60, -70.61, -70.62],
    "destination_latitude": [-33.40, -33.41, -33.42],
    "destination_longitude": [-70.50, -70.51, -70.52],
    "raw_source_col": ["A", "B", "C"],
    "debug_flag": [True, False, True],
})

schema_effective_g9 = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "trip_id": "string",
        "movement_seq": "int",
        "purpose": "categorical",
        "mode": "categorical",
        "origin_latitude": "float",
        "origin_longitude": "float",
        "destination_latitude": "float",
        "destination_longitude": "float",
        "raw_source_col": "string",  # no debería sobrevivir a prune porque no está en schema
    },
    overrides={
        "purpose": {"reasons": ["domain_extended"], "added_values": ["home"]},
        "mode": {"reasons": ["domain_extended"], "added_values": ["tram"]},
        "raw_source_col": {"reasons": ["debug_only"]},
    },
    domains_effective={
        "purpose": {
            "values": ["unknown", "work", "study", "home"],
            "extended": True,
            "added_values": ["home"],
            "unknown_value": "unknown",
            "unknown_values": [],
        },
        "mode": {
            "values": ["unknown", "bus", "metro", "walk", "tram"],
            "extended": True,
            "added_values": ["tram"],
            "unknown_value": "unknown",
            "unknown_values": [],
        },
        "raw_source_col": {
            "values": ["A", "B", "C"],
            "extended": False,
            "added_values": [],
            "unknown_value": "unknown",
            "unknown_values": [],
        },
    },
    temporal={},
    fields_effective=[],
)

In [301]:
# Tests de _select_final_columns -> con selected_fields=None, keep_extra_fields=False

options_g9_none_drop = ImportOptions(
    selected_fields=None,
    keep_extra_fields=False,
)

work_none_drop, columns_deleted_none_drop, extra_fields_kept_none_drop, issues_none_drop = _select_final_columns(
    df_g9.copy(deep=True),
    schema=schema_g9,
    options=options_g9_none_drop,
)

# Deben quedar todos los campos del schema y desaparecer extras
assert_columns_equal(
    work_none_drop,
    [
        "movement_id",
        "trip_id",
        "movement_seq",
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
        "destination_latitude",
        "destination_longitude",
    ],
    "selected_fields=None + keep_extra_fields=False",
)

assert set(columns_deleted_none_drop) == {"raw_source_col", "debug_flag"}
assert extra_fields_kept_none_drop == []

assert_issue_present(issues_none_drop, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")

show_ok("_select_final_columns - selected_fields=None, keep_extra_fields=False")

OK - _select_final_columns - selected_fields=None, keep_extra_fields=False


In [302]:
# Tests de _select_final_columns -> con selected_fields=[], keep_extra_fields=False

options_g9_empty_drop = ImportOptions(
    selected_fields=[],
    keep_extra_fields=False,
)

work_empty_drop, columns_deleted_empty_drop, extra_fields_kept_empty_drop, issues_empty_drop = _select_final_columns(
    df_g9.copy(deep=True),
    schema=schema_g9,
    options=options_g9_empty_drop,
)

# Deben quedar solo required
assert_columns_equal(
    work_empty_drop,
    ["movement_id", "trip_id", "movement_seq"],
    "selected_fields=[] + keep_extra_fields=False",
)

assert set(columns_deleted_empty_drop) == {
    "purpose", "mode",
    "origin_latitude", "origin_longitude",
    "destination_latitude", "destination_longitude",
    "raw_source_col", "debug_flag",
}
assert extra_fields_kept_empty_drop == []

assert_issue_present(issues_empty_drop, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")

show_ok("_select_final_columns - selected_fields=[], keep_extra_fields=False")

OK - _select_final_columns - selected_fields=[], keep_extra_fields=False


In [303]:
# Tests de _select_final_columns -> con subconjunto de selected_fields + keep_extra_fields=True

options_g9_subset_keep = ImportOptions(
    selected_fields=["purpose", "origin_latitude", "origin_longitude"],
    keep_extra_fields=True,
)

work_subset_keep, columns_deleted_subset_keep, extra_fields_kept_subset_keep, issues_subset_keep = _select_final_columns(
    df_g9.copy(deep=True),
    schema=schema_g9,
    options=options_g9_subset_keep,
)

# Deben quedar required + selected + extras
assert_columns_equal(
    work_subset_keep,
    [
        "movement_id",
        "trip_id",
        "movement_seq",
        "purpose",
        "origin_latitude",
        "origin_longitude",
        "raw_source_col",
        "debug_flag",
    ],
    "subset + keep_extra_fields=True",
)

assert set(columns_deleted_subset_keep) == {"mode", "destination_latitude", "destination_longitude"}
assert set(extra_fields_kept_subset_keep) == {"raw_source_col", "debug_flag"}

# No debería emitir EXTRA_FIELDS_DROPPED porque los extras se conservaron
assert_issue_absent(issues_subset_keep, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")

show_ok("_select_final_columns - subset + keep_extra_fields=True")

OK - _select_final_columns - subset + keep_extra_fields=True


In [304]:
# Tests de _select_final_columns -> con subconjunto de selected_fields + keep_extra_fields=False

options_g9_subset_drop = ImportOptions(
    selected_fields=["purpose", "origin_latitude", "origin_longitude"],
    keep_extra_fields=False,
)

work_subset_drop, columns_deleted_subset_drop, extra_fields_kept_subset_drop, issues_subset_drop = _select_final_columns(
    df_g9.copy(deep=True),
    schema=schema_g9,
    options=options_g9_subset_drop,
)

assert_columns_equal(
    work_subset_drop,
    [
        "movement_id",
        "trip_id",
        "movement_seq",
        "purpose",
        "origin_latitude",
        "origin_longitude",
    ],
    "subset + keep_extra_fields=False",
)

assert set(columns_deleted_subset_drop) == {
    "mode",
    "destination_latitude",
    "destination_longitude",
    "raw_source_col",
    "debug_flag",
}
assert extra_fields_kept_subset_drop == []

assert_issue_present(issues_subset_drop, "IMP.OPTIONS.EXTRA_FIELDS_DROPPED")

show_ok("_select_final_columns - subset + keep_extra_fields=False")

OK - _select_final_columns - subset + keep_extra_fields=False


In [305]:
# Tests de _final_required_check -> caso feliz

df_required_ok = pd.DataFrame({
    "movement_id": ["m0", "m1"],
    "trip_id": ["t0", "t1"],
    "movement_seq": pd.Series([0, 0], dtype="Int64"),
    "purpose": ["work", "study"],
})

# No debería lanzar excepción
_final_required_check(
    df_required_ok,
    schema=schema_g9,
    single_stage=True,
    strict=False,
)

show_ok("_final_required_check - caso feliz")

OK - _final_required_check - caso feliz


In [306]:
# Tests de _final_required_check -> required que desaparece por error

df_missing_required_final = pd.DataFrame({
    "movement_id": ["m0", "m1"],
    "trip_id": ["t0", "t1"],
    # falta movement_seq
    "purpose": ["work", "study"],
})

try:
    _final_required_check(
        df_missing_required_final,
        schema=schema_g9,
        single_stage=True,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por missing required final"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("_final_required_check - missing required final")

OK - _final_required_check - missing required final


In [307]:
# Tests de _final_required_check -> single_stage=False no debe exigir movement_seq extra si no corresponde

df_single_false = pd.DataFrame({
    "movement_id": ["m0", "m1"],
    "trip_id": ["t0", "t1"],
    "movement_seq": pd.Series([0, 0], dtype="Int64"),
})

# Como schema_g9 sí lo tiene como required, aquí igual debe estar presente.
_final_required_check(
    df_single_false,
    schema=schema_g9,
    single_stage=False,
    strict=False,
)

show_ok("_final_required_check - single_stage=False")

OK - _final_required_check - single_stage=False


In [308]:
# Tests de _prune_schema_effective

df_pruned = pd.DataFrame({
    "movement_id": ["m0", "m1"],
    "trip_id": ["t0", "t1"],
    "movement_seq": pd.Series([0, 0], dtype="Int64"),
    "purpose": ["work", "home"],
    "origin_latitude": [-33.45, -33.46],
    "origin_longitude": [-70.60, -70.61],
    "raw_source_col": ["A", "B"],  # extra, no debería considerarse final_schema_fields
})

schema_effective_pruned = TripSchemaEffective(
    dtype_effective=dict(schema_effective_g9.dtype_effective),
    overrides={k: dict(v) for k, v in schema_effective_g9.overrides.items()},
    domains_effective={k: dict(v) for k, v in schema_effective_g9.domains_effective.items()},
    temporal={},
    fields_effective=[],
)

pruned = _prune_schema_effective(
    schema_effective_pruned,
    df=df_pruned,
    schema=schema_g9,
)

# fields_effective debe contener solo campos del schema presentes en df
assert pruned.fields_effective == [
    "movement_id",
    "trip_id",
    "movement_seq",
    "purpose",
    "origin_latitude",
    "origin_longitude",
]

# dtype_effective podado
assert set(pruned.dtype_effective.keys()) == {
    "movement_id",
    "trip_id",
    "movement_seq",
    "purpose",
    "origin_latitude",
    "origin_longitude",
}

# overrides podado: mode y raw_source_col deben desaparecer
assert set(pruned.overrides.keys()) == {"purpose"}

# domains_effective podado: mode y raw_source_col deben desaparecer
assert set(pruned.domains_effective.keys()) == {"purpose"}

assert_json_safe(pruned.to_dict(), "pruned_schema_effective")

show_ok("_prune_schema_effective")

OK - _prune_schema_effective


In [309]:
# Test integrado pequeño del Grupo 9: select + prune

options_g9_integrated = ImportOptions(
    selected_fields=["purpose", "origin_latitude"],
    keep_extra_fields=False,
)

work_integrated, columns_deleted_integrated, extra_fields_kept_integrated, issues_integrated = _select_final_columns(
    df_g9.copy(deep=True),
    schema=schema_g9,
    options=options_g9_integrated,
)

schema_effective_integrated = TripSchemaEffective(
    dtype_effective=dict(schema_effective_g9.dtype_effective),
    overrides={k: dict(v) for k, v in schema_effective_g9.overrides.items()},
    domains_effective={k: dict(v) for k, v in schema_effective_g9.domains_effective.items()},
    temporal={},
    fields_effective=[],
)

pruned_integrated = _prune_schema_effective(
    schema_effective_integrated,
    df=work_integrated,
    schema=schema_g9,
)

assert_columns_equal(
    work_integrated,
    ["movement_id", "trip_id", "movement_seq", "purpose", "origin_latitude"],
    "integrated select+prune columns",
)

assert pruned_integrated.fields_effective == [
    "movement_id",
    "trip_id",
    "movement_seq",
    "purpose",
    "origin_latitude",
]

assert set(pruned_integrated.domains_effective.keys()) == {"purpose"}
assert set(pruned_integrated.overrides.keys()) == {"purpose"}

show_ok("Grupo 9 - test integrado pequeño")

OK - Grupo 9 - test integrado pequeño


In [310]:
show_ok("Grupo 9 completo")

OK - Grupo 9 completo


## Sección 6. Metadata, report y trazabilidad
### Grupo 10 - metadata, report y trazabilidad

En esta sección se prueban las helpers que construyen la evidencia operacional del import.

Objetivo:
- verificar que la metadata quede bien formada y serializable,
- comprobar que `dataset_id` se genere y se registre,
- revisar que el evento `import_trips` quede embebido correctamente,
- y asegurar que el `ImportReport` sea coherente con los issues y el resumen operativo.

In [311]:
# Fixtures comunes del Grupo 10

schema_g10 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", "study"], extendable=True, aliases=None),
        ),
        "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime", required=False),
    },
    required=["movement_id"],
    semantic_rules=None,
)

schema_effective_g10 = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "trip_id": "string",
        "movement_seq": "int",
        "purpose": "categorical",
        "origin_time_utc": "datetime",
    },
    overrides={
        "purpose": {"reasons": ["domain_extended"], "added_values": ["home"]},
        "origin_time_utc": {"reasons": ["datetime_normalized"], "status": "string_tzaware_to_utc"},
    },
    domains_effective={
        "purpose": {
            "values": ["unknown", "work", "study", "home"],
            "extended": True,
            "added_values": ["home"],
            "unknown_value": "unknown",
            "unknown_values": [],
        },
    },
    temporal={"tier": "tier_1"},
    fields_effective=["movement_id", "trip_id", "movement_seq", "purpose", "origin_time_utc"],
)

field_corr_g10 = {"purpose": "motivo"}
value_corr_g10 = {"purpose": {"trabajo": "work"}}
domains_effective_g10 = schema_effective_g10.domains_effective
domains_extended_g10 = ["purpose"]
extra_fields_kept_g10 = ["raw_source_col"]
h3_meta_g10 = {
    "resolution": 8,
    "source_fields": [["origin_latitude", "origin_longitude"]],
    "derived_fields": ["origin_h3_index"],
}
provenance_g10 = {
    "source_name": "eod_santiago_2012",
    "created_by_op": "import_trips",
    "created_at_utc": "2026-03-20T15:00:00+00:00",
}
event_import_g10 = {
    "op": "import_trips",
    "ts_utc": "2026-03-20T15:00:00+00:00",
    "parameters": {
        "keep_extra_fields": True,
        "selected_fields": ["purpose"],
        "strict": False,
        "strict_domains": False,
        "single_stage": True,
        "h3_resolution": 8,
        "source_name": "eod_santiago_2012",
    },
    "summary": {
        "input_rows": 10,
        "output_rows": 10,
        "rows_dropped": 0,
        "n_fields_mapped": 1,
        "n_domain_mappings_applied": 1,
        "columns_added": ["movement_id", "trip_id", "movement_seq"],
        "columns_deleted": [],
        "domains_extended_count": 1,
        "temporal_tier": "tier_1",
        "temporal_notes": "tier_1_datetime_normalized",
    },
    "issues_summary": {
        "counts": {"info": 1, "warning": 1},
        "by_code": {"DOM.EXTENSION.APPLIED": 1, "IMP.TEMPORAL.TIER_LIMITED": 1},
    },
}

issues_g10_base = [
    Issue(level="info", code="DOM.EXTENSION.APPLIED", message="domain extended"),
    Issue(level="warning", code="IMP.TEMPORAL.TIER_LIMITED", message="tier limited"),
]

In [312]:
# Tests de _build_import_metadata -> metadata mínima feliz

issues_meta_ok = list(issues_g10_base)

metadata_ok = _build_import_metadata(
    schema=schema_g10,
    schema_effective=schema_effective_g10,
    field_correspondence_applied=field_corr_g10,
    value_correspondence_applied=value_corr_g10,
    domains_effective=domains_effective_g10,
    domains_extended=domains_extended_g10,
    extra_fields_kept=extra_fields_kept_g10,
    h3_meta=h3_meta_g10,
    provenance=provenance_g10,
    temporal_tier_detected="tier_1",
    temporal_fields_present=["origin_time_utc", "destination_time_utc"],
    datetime_normalization_status_by_field={
        "origin_time_utc": {"status": "string_tzaware_to_utc", "n_nat": 0},
        "destination_time_utc": {"status": "string_tzaware_to_utc", "n_nat": 0},
    },
    datetime_normalization_stats_t2 = {},
    source_timezone_used="UTC",
    event_import=event_import_g10,
    issues=issues_meta_ok,
    strict=False,
)

# estructura mínima esperada
assert "dataset_id" in metadata_ok
assert isinstance(metadata_ok["dataset_id"], str)
assert metadata_ok["dataset_id"].startswith("tripds_")
assert metadata_ok["is_validated"] is False

assert metadata_ok["schema"]["version"] == "0.1.0"
assert metadata_ok["schema_effective"]["fields_effective"] == schema_effective_g10.fields_effective

assert metadata_ok["mappings"]["field_correspondence"] == field_corr_g10
assert metadata_ok["mappings"]["value_correspondence"] == value_corr_g10

assert metadata_ok["domains_effective"] == domains_effective_g10
assert metadata_ok["domains_extended"] == domains_extended_g10
assert metadata_ok["extra_fields_kept"] == extra_fields_kept_g10

assert metadata_ok["events"][0]["op"] == "import_trips"
assert metadata_ok["provenance"] == provenance_g10

assert metadata_ok["temporal"]["tier"] == "tier_1"
assert metadata_ok["temporal"]["fields_present"] == ["origin_time_utc", "destination_time_utc"]
assert "normalization" in metadata_ok["temporal"]
assert metadata_ok["temporal"]["source_timezone_used"] == "UTC"

assert metadata_ok["h3"] == h3_meta_g10

# se debe haber agregado el issue de dataset_id generado
assert_issue_present(issues_meta_ok, "IMP.METADATA.DATASET_ID_CREATED")

assert_json_safe(metadata_ok, "metadata_ok")

show_ok("_build_import_metadata - caso mínimo feliz")

OK - _build_import_metadata - caso mínimo feliz


In [313]:
# Tests de _build_import_metadata -> tier_2 registrado sin normalization de datetime

issues_meta_t2 = []

metadata_t2 = _build_import_metadata(
    schema=schema_g10,
    schema_effective=schema_effective_g10,
    field_correspondence_applied={},
    value_correspondence_applied={},
    domains_effective={},
    domains_extended=[],
    extra_fields_kept=[],
    h3_meta=None,
    provenance=provenance_g10,
    temporal_tier_detected="tier_2",
    temporal_fields_present=["origin_time_local_hhmm", "destination_time_local_hhmm"],
    datetime_normalization_status_by_field={},
    datetime_normalization_stats_t2={
        "origin_time_local_hhmm" : {"n_total": 5, "n_invalid": 1, "n_na": 2}, 
        "destination_time_local_hhmm" : {"n_total": 5, "n_invalid": 2, "n_na": 1}
    },
    source_timezone_used=None,
    event_import=event_import_g10,
    issues=issues_meta_t2,
    strict=False,
)

assert metadata_t2["temporal"]["tier"] == "tier_2"
assert metadata_t2["temporal"]["fields_present"] == ["origin_time_local_hhmm", "destination_time_local_hhmm"]
assert "source_timezone_used" not in metadata_t2["temporal"]
assert "h3" not in metadata_t2

assert metadata_t2["is_validated"] is False
assert_issue_present(issues_meta_t2, "IMP.METADATA.DATASET_ID_CREATED")

show_ok("_build_import_metadata - tier_2")

OK - _build_import_metadata - tier_2


In [314]:
# Test de _build_import_metadata -> provenance no serializable con strict=False
issues_meta_nonserial = []

try:
    _build_import_metadata(
        schema=schema_g10,
        schema_effective=schema_effective_g10,
        field_correspondence_applied={},
        value_correspondence_applied={},
        domains_effective={},
        domains_extended=[],
        extra_fields_kept=[],
        h3_meta=None,
        provenance={"bad": {1, 2, 3}},
        temporal_tier_detected="tier_3",
        temporal_fields_present=[],
        datetime_normalization_status_by_field={},
        datetime_normalization_stats_t2={},
        source_timezone_used=None,
        event_import=event_import_g10,
        issues=issues_meta_nonserial,
        strict=False,
    )
    assert False, "Debió lanzar PylondrinaImportError por provenance no serializable"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "PRV.INPUT.NOT_JSON_SERIALIZABLE")

show_ok("_build_import_metadata - provenance no serializable")

OK - _build_import_metadata - provenance no serializable


In [315]:
# Test de _build_import_metadata -> provenance no serializable con strict=True

issues_meta_nonserial_strict = []

try:
    _build_import_metadata(
        schema=schema_g10,
        schema_effective=schema_effective_g10,
        field_correspondence_applied={},
        value_correspondence_applied={},
        domains_effective={},
        domains_extended=[],
        extra_fields_kept=[],
        h3_meta=None,
        provenance={"bad": {1, 2, 3}},   # set no serializable
        temporal_tier_detected="tier_3",
        temporal_fields_present=[],
        datetime_normalization_status_by_field={},
        datetime_normalization_stats_t2={},
        source_timezone_used=None,
        event_import=event_import_g10,
        issues=issues_meta_nonserial_strict,
        strict=True,
    )
    assert False, "Debió lanzar PylondrinaImportError por provenance no serializable con strict=True"
except PylondrinaImportError as error:
    assert_issue_present(error.issues, "PRV.INPUT.NOT_JSON_SERIALIZABLE")

show_ok("_build_import_metadata - provenance no serializable con strict=True")

OK - _build_import_metadata - provenance no serializable con strict=True


In [316]:
# Tests de _build_import_report -> caso feliz sin errores

metadata_for_report_ok = {
    "dataset_id": "tripds_dummy_001",
    "is_validated": False,
    "temporal": {"tier": "tier_1"},
}

issues_report_ok = [
    Issue(level="info", code="DOM.EXTENSION.APPLIED", message="domain extended"),
    Issue(level="warning", code="IMP.TEMPORAL.TIER_LIMITED", message="tier limited"),
]

report_ok = _build_import_report(
    issues=issues_report_ok,
    field_correspondence_applied=field_corr_g10,
    value_correspondence_applied=value_corr_g10,
    schema_version="0.1.0",
    source_name="eod_santiago_2012",
    dataset_id="tripds_dummy_001",
    parameters_effective={
        "keep_extra_fields": True,
        "selected_fields": ["purpose"],
        "strict": False,
        "strict_domains": False,
        "single_stage": True,
    },
    rows_in=10,
    rows_out=10,
    n_fields_mapped=1,
    n_domain_mappings_applied=1,
    metadata=metadata_for_report_ok,
)

assert report_ok.ok is True
assert report_ok.summary == {
    "rows_in": 10,
    "rows_out": 10,
    "n_fields_mapped": 1,
    "n_domain_mappings_applied": 1,
}
assert report_ok.parameters["single_stage"] is True
assert report_ok.field_correspondence == field_corr_g10
assert report_ok.value_correspondence == value_corr_g10
assert report_ok.schema_version == "0.1.0"

assert report_ok.metadata["schema_version"] == "0.1.0"
assert report_ok.metadata["dataset_id"] == "tripds_dummy_001"
assert report_ok.metadata["source_name"] == "eod_santiago_2012"
assert report_ok.metadata["summary"]["rows_in"] == 10
assert report_ok.metadata["summary"]["rows_out"] == 10
assert report_ok.metadata["metadata"] == metadata_for_report_ok

assert_json_safe(report_ok.metadata, "report_ok.metadata")

show_ok("_build_import_report - caso feliz")

OK - _build_import_report - caso feliz


In [317]:
# Tests de _build_import_report -> ok=False cuando hay issue de nivel error

issues_report_error = [
    Issue(level="warning", code="IMP.TEMPORAL.TIER_LIMITED", message="tier limited"),
    Issue(level="error", code="IMP.INPUT.MISSING_REQUIRED_FIELD", message="missing required"),
]

report_error = _build_import_report(
    issues=issues_report_error,
    field_correspondence_applied={},
    value_correspondence_applied={},
    schema_version="0.1.0",
    source_name="eod_santiago_2012",
    dataset_id="tripds_dummy_002",
    parameters_effective={"strict": False},
    rows_in=5,
    rows_out=4,
    n_fields_mapped=0,
    n_domain_mappings_applied=0,
    metadata={"dataset_id": "tripds_dummy_002", "is_validated": False},
)

assert report_error.ok is False
assert len(report_error.issues) == 2
assert report_error.summary["rows_in"] == 5
assert report_error.summary["rows_out"] == 4

show_ok("_build_import_report - ok=False con issue error")

OK - _build_import_report - ok=False con issue error


In [318]:
# Test integrado pequeño del Grupo 10: metadata + report

issues_integrated_g10 = [
    Issue(level="warning", code="IMP.TEMPORAL.TIER_LIMITED", message="tier limited")
]

metadata_integrated_g10 = _build_import_metadata(
    schema=schema_g10,
    schema_effective=schema_effective_g10,
    field_correspondence_applied=field_corr_g10,
    value_correspondence_applied=value_corr_g10,
    domains_effective=domains_effective_g10,
    domains_extended=domains_extended_g10,
    extra_fields_kept=[],
    h3_meta=None,
    provenance=provenance_g10,
    temporal_tier_detected="tier_3",
    temporal_fields_present=[],
    datetime_normalization_status_by_field={},
    datetime_normalization_stats_t2={},
    source_timezone_used=None,
    event_import=event_import_g10,
    issues=issues_integrated_g10,
    strict=False,
)

dataset_id_integrated_g10 = metadata_integrated_g10["dataset_id"]

report_integrated_g10 = _build_import_report(
    issues=issues_integrated_g10,
    field_correspondence_applied=field_corr_g10,
    value_correspondence_applied=value_corr_g10,
    schema_version=schema_g10.version,
    source_name=provenance_g10["source_name"],
    dataset_id=dataset_id_integrated_g10,
    parameters_effective=event_import_g10["parameters"],
    rows_in=10,
    rows_out=10,
    n_fields_mapped=1,
    n_domain_mappings_applied=1,
    metadata=metadata_integrated_g10,
)

assert report_integrated_g10.ok is True
assert report_integrated_g10.metadata["dataset_id"] == dataset_id_integrated_g10
assert report_integrated_g10.metadata["metadata"]["dataset_id"] == dataset_id_integrated_g10
assert report_integrated_g10.metadata["metadata"]["is_validated"] is False
assert report_integrated_g10.metadata["metadata"]["events"][0]["op"] == "import_trips"
assert report_integrated_g10.metadata["metadata"]["temporal"]["tier"] == "tier_3"

show_ok("Grupo 10 - test integrado pequeño")

OK - Grupo 10 - test integrado pequeño


In [319]:
show_ok("Grupo 10 completo")

OK - Grupo 10 completo


## Sección 7. Smoke tests integrados
### Grupo 11 - smoke tests de `import_trips_from_dataframe`

En esta sección se prueban caminos integrados pequeños de `import_trips_from_dataframe()`.

Objetivo:
- comprobar que la integración básica de helpers no esté rota,
- verificar caminos felices y abortos estructurales esperados,
- y congelar casos clave: correspondencias, dominios extendibles, Tier 2 y `single_stage=True`.

In [320]:
# Smoke test 11.1 - happy path simple

schema_smoke_simple = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "unknown"], extendable=False, aliases=None),
        ),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_smoke_simple = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "purpose": ["work", "study", "work"],
})

tripds_simple, report_simple = import_trips_from_dataframe(
    df_smoke_simple,
    schema_smoke_simple,
    source_name="smoke_simple",
    options=ImportOptions(strict=False, strict_domains=False, single_stage=False),
)

print(tripds_simple)
display(report_simple)

assert isinstance(tripds_simple, TripDataset)
assert isinstance(report_simple, ImportReport)
assert report_simple.ok is True

assert_columns_equal(
    tripds_simple.data,
    ["user_id", "purpose", "movement_id"],
    "smoke simple columns",
)

assert tripds_simple.data["user_id"].tolist() == ["u1", "u2", "u3"]
assert tripds_simple.data["purpose"].tolist() == ["work", "study", "work"]
assert tripds_simple.data["movement_id"].tolist() == ["m0", "m1", "m2"]

assert tripds_simple.metadata["is_validated"] is False
assert tripds_simple.metadata["temporal"]["tier"] == "tier_3"
assert tripds_simple.metadata["events"][0]["op"] == "import_trips"

assert_issue_present(report_simple.issues, "IMP.ID.MOVEMENT_ID_CREATED")

show_ok("Smoke 11.1 - happy path simple")

TripDataset
-----------
shape: (3, 3)
schema_version: 0.1.0
is_validated: False
columns: ['movement_id', 'user_id', 'purpose']

data:
  movement_id user_id purpose
0          m0      u1    work
1          m1      u2   study
2          m2      u3    work

schema:
TripSchema(
{'version': '0.1.0',
 'required': ['user_id'],
 'semantic_rules': None,
 'fields': {'user_id': {'name': 'user_id', 'dtype': 'string', 'required': True, 'constraints': None, 'domain': None},
            'purpose': {'name': 'purpose',
                        'dtype': 'categorical',
                        'required': False,
                        'constraints': None,
                        'domain': {'values': ['work', 'study', 'unknown'], 'extendable': False, 'aliases': None}},
            'movement_id': {'name': 'movement_id',
                            'dtype': 'string',
                            'required': False,
                            'constraints': None,
                            'domain': None}}}
)

ImportReport
------------
ok: True
schema_version: 0.1.0
issues_count: 3

issues:
  [1] WARNING - IMP.TEMPORAL.TIER_LIMITED
      message: El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.
      details:
      {'temporal_tier': 'tier_3',
       'fields_present': [],
       'note': 'limited_temporal_capabilities'}
  [2] INFO - IMP.ID.MOVEMENT_ID_CREATED
      message: Se generó movement_id porque no venía presente en la fuente.
      field: movement_id
      details:
      {'field': 'movement_id', 'action': 'generated'}
  [3] INFO - IMP.METADATA.DATASET_ID_CREATED
      message: Se generó dataset_id para el dataset importado: 'tripds_99525187ef7e44aeac879fee6134be9d'.
      details:
      {'dataset_id': 'tripds_99525187ef7e44aeac879fee6134be9d',
       'generator': 'uuid4_hex',
       'stored_in': 'metadata.dataset_id'}

summary:
{'rows_in': 3, 'rows_out': 3, 'n_fields_mapped': 0, 'n_domain_mappings_applied': 0}

pa

OK - Smoke 11.1 - happy path simple


In [321]:
# Smoke test 11.2 - happy path con correspondencias

schema_smoke_corr = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", "study", "unknown"], extendable=False, aliases=None),
        ),
        "mode": FieldSpec(
            name="mode",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["bus", "metro", "walk", "unknown"], extendable=False, aliases=None),
        ),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_smoke_corr = pd.DataFrame({
    "uid": ["u1", "u2"],
    "motivo": ["trabajo", "estudio"],
    "modo": ["autobus", "metro"],
    "raw_extra": ["A", "B"],
})

tripds_corr, report_corr = import_trips_from_dataframe(
    df_smoke_corr,
    schema_smoke_corr,
    source_name="smoke_corr",
    options=ImportOptions(strict=False, strict_domains=False, keep_extra_fields=True),
    field_correspondence={
        "user_id": "uid",
        "purpose": "motivo",
        "mode": "modo",
    },
    value_correspondence={
        "purpose": {"trabajo": "work", "estudio": "study"},
        "mode": {"autobus": "bus"},
    },
)

assert report_corr.ok is True

assert_columns_equal(
    tripds_corr.data,
    ["user_id", "purpose", "mode", "raw_extra", "movement_id"],
    "smoke correspondences columns",
)

assert tripds_corr.data["user_id"].tolist() == ["u1", "u2"]
assert tripds_corr.data["purpose"].tolist() == ["work", "study"]
assert tripds_corr.data["mode"].tolist() == ["bus", "metro"]

assert tripds_corr.field_correspondence == {
    "user_id": "uid",
    "purpose": "motivo",
    "mode": "modo",
}
assert tripds_corr.value_correspondence == {
    "purpose": {"trabajo": "work", "estudio": "study"},
    "mode": {"autobus": "bus"},
}

show_ok("Smoke 11.2 - happy path con correspondencias")
display(tripds_corr.data)
display(report_corr.issues)

OK - Smoke 11.2 - happy path con correspondencias


,movement_id,user_id,purpose,mode,raw_extra
0,m0,u1,work,bus,A
1,m1,u2,study,metro,B


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_3', 'fields_present': [], 'note': 'limited_temporal_capabilities'}),
 Issue(level='info', code='IMP.ID.MOVEMENT_ID_CREATED', message='Se generó movement_id porque no venía presente en la fuente.', field='movement_id', source_field=None, row_count=None, details={'field': 'movement_id', 'action': 'generated'}),
 Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó dataset_id para el dataset importado: 'tripds_026484e8e7c84779851b1056b203839d'.", field=None, source_field=None, row_count=None, details={'dataset_id': 'tripds_026484e8e7c84779851b1056b203839d', 'generator': 'uuid4_hex', 'stored_in': 'metadata.dataset_id'})]

In [322]:
# Smoke test 11.3 - out-of-domain extendible

schema_smoke_extend = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(values=["work", "study"], extendable=True, aliases=None),
        ),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_smoke_extend = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "purpose": ["work", "home", "study"],
})

tripds_extend, report_extend = import_trips_from_dataframe(
    df_smoke_extend,
    schema_smoke_extend,
    source_name="smoke_extend",
    options=ImportOptions(strict=False, strict_domains=False),
)

assert report_extend.ok is True

assert tripds_extend.data["purpose"].tolist() == ["work", "home", "study"]

assert "purpose" in tripds_extend.metadata["domains_effective"]
assert tripds_extend.metadata["domains_effective"]["purpose"]["extended"] is True
assert tripds_extend.metadata["domains_effective"]["purpose"]["added_values"] == ["home"]
assert "purpose" in tripds_extend.metadata["domains_extended"]

assert_issue_present(report_extend.issues, "DOM.EXTENSION.APPLIED")

show_ok("Smoke 11.3 - out-of-domain extendible")
display(tripds_extend.data)
display(report_extend.issues)

OK - Smoke 11.3 - out-of-domain extendible


,movement_id,user_id,purpose
0,m0,u1,work
1,m1,u2,home
2,m2,u3,study


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_3', 'fields_present': [], 'note': 'limited_temporal_capabilities'}),
 Issue(level='info', code='DOM.EXTENSION.APPLIED', message="Se extendió el dominio de 'purpose' con 1 valores nuevos.", field='purpose', source_field=None, row_count=None, details={'field': 'purpose', 'n_added': 1, 'added_values_sample': ['home'], 'added_values_total': 1, 'policy': 'extendable_non_strict', 'action': 'extended_domain'}),
 Issue(level='info', code='IMP.ID.MOVEMENT_ID_CREATED', message='Se generó movement_id porque no venía presente en la fuente.', field='movement_id', source_field=None, row_count=None, details={'field': 'movement_id', 'action': 'generated'}),
 Issue(level='info', code='IMP.METADATA.DATASET_ID_CREATED', message="Se generó 

In [323]:
# Smoke test 11.4 - missing required fatal

schema_smoke_missing = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "purpose": FieldSpec(name="purpose", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_smoke_missing = pd.DataFrame({
    "purpose": ["work", "study"],
})

try:
    import_trips_from_dataframe(
        df_smoke_missing,
        schema_smoke_missing,
        source_name="smoke_missing",
        options=ImportOptions(strict=False),
    )
    assert False, "Debió lanzar PylondrinaImportError por missing required"
except PylondrinaImportError as error:
    print("Mensaje del error: ", error.message)
    assert_issue_present(error.issues, "IMP.INPUT.MISSING_REQUIRED_FIELD")

show_ok("Smoke 11.4 - missing required fatal")

Mensaje del error:  Faltan campos obligatorios para importar según el TripSchema: ['user_id'].
OK - Smoke 11.4 - missing required fatal


In [324]:
# Smoke test 11.5 - Tier 2

schema_smoke_t2 = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string", required=False),
        "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string", required=False),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=False),
    },
    required=["user_id"],
    semantic_rules=None,
)

df_smoke_t2 = pd.DataFrame({
    "user_id": ["u1", "u2", "u3"],
    "origin_time_local_hhmm": ["08:00", "24:00", "07:05"],
    "destination_time_local_hhmm": ["08:30", "13:00", "99:99"],
})

tripds_t2, report_t2 = import_trips_from_dataframe(
    df_smoke_t2,
    schema_smoke_t2,
    source_name="smoke_t2",
    options=ImportOptions(strict=False, single_stage=False),
)

assert report_t2.ok is True

assert tripds_t2.metadata["temporal"]["tier"] == "tier_2"
assert tripds_t2.metadata["temporal"]["fields_present"] == [
    "origin_time_local_hhmm",
    "destination_time_local_hhmm",
]

print(tripds_t2.metadata["temporal"])
assert "normalization" in tripds_t2.metadata["temporal"]

# válidos preservados
assert tripds_t2.data.loc[0, "origin_time_local_hhmm"] == "08:00"
assert tripds_t2.data.loc[0, "destination_time_local_hhmm"] == "08:30"
assert tripds_t2.data.loc[2, "origin_time_local_hhmm"] == "07:05"

# inválidos -> NA
assert pd.isna(tripds_t2.data.loc[1, "origin_time_local_hhmm"])   # 24:00
assert pd.isna(tripds_t2.data.loc[2, "destination_time_local_hhmm"])  # 99:99

assert_issue_present(report_t2.issues, "IMP.TEMPORAL.TIER_LIMITED")
assert_issue_present(report_t2.issues, "IMP.TYPE.COERCE_PARTIAL")

show_ok("Smoke 11.5 - Tier 2")
display(tripds_t2.data)
display(report_t2.issues)

{'tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'normalization': {'origin_time_local_hhmm': {'n_total': 3, 'n_invalid': 1, 'n_na': 1}, 'destination_time_local_hhmm': {'n_total': 3, 'n_invalid': 1, 'n_na': 1}}}
OK - Smoke 11.5 - Tier 2


,movement_id,user_id,origin_time_local_hhmm,destination_time_local_hhmm
0,m0,u1,08:00,08:30
1,m1,u2,<NA>,13:00
2,m2,u3,07:05,<NA>


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_2); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_2', 'fields_present': ['origin_time_local_hhmm', 'destination_time_local_hhmm'], 'note': 'limited_temporal_capabilities'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'origin_time_local_hhmm' falló en algunas filas (33.3%); se marcarán como nulos para validación posterior.", field='origin_time_local_hhmm', source_field=None, row_count=None, details={'field': 'origin_time_local_hhmm', 'dtype_expected': 'string_hhmm', 'parse_fail_count': 1, 'total_count': 3, 'fail_rate': 0.3333333333333333, 'fallback': 'set_null', 'action': 'continue'}),
 Issue(level='warning', code='IMP.TYPE.COERCE_PARTIAL', message="La conversión mínima del campo 'destination_time_local_hhmm' falló en 

In [325]:
# Smoke test 11.6 - single_stage=True

schema_smoke_single = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "purpose": FieldSpec(name="purpose", dtype="string", required=False),
    },
    required=["user_id", "movement_id", "trip_id", "movement_seq"],
    semantic_rules=None,
)

df_smoke_single = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "purpose": ["work", "study"],
})

tripds_single, report_single = import_trips_from_dataframe(
    df_smoke_single,
    schema_smoke_single,
    source_name="smoke_single",
    options=ImportOptions(strict=False, single_stage=True),
)

assert report_single.ok is True

assert_columns_equal(
    tripds_single.data,
    ["user_id", "movement_id", "trip_id", "movement_seq", "purpose"],
    "smoke single_stage columns",
)

assert tripds_single.data["movement_id"].tolist() == ["m0", "m1"]
assert tripds_single.data["trip_id"].tolist() == ["m0", "m1"]
assert tripds_single.data["movement_seq"].tolist() == [0, 0]
assert_dtype(tripds_single.data, "movement_seq", "Int64")

assert_issue_present(report_single.issues, "IMP.ID.MOVEMENT_ID_CREATED")
assert_issue_present(report_single.issues, "IMP.ID.TRIP_ID_CREATED")
assert_issue_present(report_single.issues, "IMP.ID.MOVEMENT_SEQ_CREATED")

show_ok("Smoke 11.6 - single_stage=True")
display(tripds_single.data)
display(report_single.issues)

OK - Smoke 11.6 - single_stage=True


,movement_id,trip_id,movement_seq,user_id,purpose
0,m0,m0,0,u1,work
1,m1,m1,0,u2,study


[Issue(level='warning', code='IMP.TEMPORAL.TIER_LIMITED', message='El dataset fue importado con capacidad temporal limitada (tier_3); algunas operaciones posteriores quedarán restringidas.', field=None, source_field=None, row_count=None, details={'temporal_tier': 'tier_3', 'fields_present': [], 'note': 'limited_temporal_capabilities'}),
 Issue(level='info', code='IMP.ID.MOVEMENT_ID_CREATED', message='Se generó movement_id porque no venía presente en la fuente.', field='movement_id', source_field=None, row_count=None, details={'field': 'movement_id', 'action': 'generated'}),
 Issue(level='info', code='IMP.ID.TRIP_ID_CREATED', message='Se generó trip_id a partir de movement_id porque single_stage=True y no venía en la fuente.', field='trip_id', source_field=None, row_count=None, details={'field': 'trip_id', 'action': 'generated_from_movement_id'}),
 Issue(level='info', code='IMP.ID.MOVEMENT_SEQ_CREATED', message='Se generó movement_seq=0 porque single_stage=True y no venía en la fuente.'

## Bug al tener options.single_stage=True (arreglado)

### Test 1 - single_stage=True sin campos derivables en schema

In [326]:
from pylondrina.importing import import_trips_from_dataframe, ImportOptions
from pylondrina.schema import TripSchema, FieldSpec

schema = TripSchema(
    version="test-1",
    fields={
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
    },
    required=[
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

df = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_longitude": [-70.65, -70.66],
    "origin_latitude": [-33.45, -33.46],
    "destination_longitude": [-70.61, -70.62],
    "destination_latitude": [-33.41, -33.42],
})

opts = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

tripds, report = import_trips_from_dataframe(
    df,
    schema=schema,
    source_name="test_single_stage_no_ids_in_schema",
    options=opts,
    h3_resolution=8,
)

print(tripds.data.columns.tolist())
display(tripds.data.head())

['movement_id', 'trip_id', 'movement_seq', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude']


,movement_id,trip_id,movement_seq,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude
0,m0,m0,0,u1,-70.65,-33.45,-70.61,-33.41
1,m1,m1,0,u2,-70.66,-33.46,-70.62,-33.42


### Test 2 - single_stage=True con movement_id mapeado por correspondence

In [327]:
schema = TripSchema(
    version="test-2",
    fields={
        "movement_id": FieldSpec("movement_id", "string", required=True),
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
    },
    required=[
        "movement_id",
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

df = pd.DataFrame({
    "Viaje": ["v1", "v2"],
    "Persona": ["p1", "p2"],
    "origin_longitude": [-70.65, -70.66],
    "origin_latitude": [-33.45, -33.46],
    "destination_longitude": [-70.61, -70.62],
    "destination_latitude": [-33.41, -33.42],
})

opts = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=True,
    source_timezone=None,
)

field_corr = {
    "movement_id": "Viaje",
    "user_id": "Persona",
}

tripds, report = import_trips_from_dataframe(
    df,
    schema=schema,
    source_name="test_single_stage_with_movement_mapping",
    options=opts,
    field_correspondence=field_corr,
    h3_resolution=8,
)

print(tripds.data.columns.tolist())
display(tripds.data.head())

['movement_id', 'trip_id', 'movement_seq', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude']


,movement_id,trip_id,movement_seq,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude
0,v1,v1,0,p1,-70.65,-33.45,-70.61,-33.41
1,v2,v2,0,p2,-70.66,-33.46,-70.62,-33.42


### Test 3 - single_stage=False no debe forzar derivables extra

In [328]:
schema = TripSchema(
    version="test-3",
    fields={
        "user_id": FieldSpec("user_id", "string", required=True),
        "origin_longitude": FieldSpec("origin_longitude", "float", required=True),
        "origin_latitude": FieldSpec("origin_latitude", "float", required=True),
        "destination_longitude": FieldSpec("destination_longitude", "float", required=True),
        "destination_latitude": FieldSpec("destination_latitude", "float", required=True),
    },
    required=[
        "user_id",
        "origin_longitude",
        "origin_latitude",
        "destination_longitude",
        "destination_latitude",
    ],
)

df = pd.DataFrame({
    "user_id": ["u1", "u2"],
    "origin_longitude": [-70.65, -70.66],
    "origin_latitude": [-33.45, -33.46],
    "destination_longitude": [-70.61, -70.62],
    "destination_latitude": [-33.41, -33.42],
})

opts = ImportOptions(
    keep_extra_fields=False,
    selected_fields=None,
    strict=False,
    strict_domains=False,
    single_stage=False,
    source_timezone=None,
)

tripds, report = import_trips_from_dataframe(
    df,
    schema=schema,
    source_name="test_non_single_stage",
    options=opts,
    h3_resolution=8,
)

print(tripds.data.columns.tolist())
display(tripds.data.head())

['movement_id', 'user_id', 'origin_longitude', 'origin_latitude', 'destination_longitude', 'destination_latitude']


,movement_id,user_id,origin_longitude,origin_latitude,destination_longitude,destination_latitude
0,m0,u1,-70.65,-33.45,-70.61,-33.41
1,m1,u2,-70.66,-33.46,-70.62,-33.42
